<a href="https://colab.research.google.com/github/Drugpurchasing/Shift/blob/main/Shift_Sch_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

import numpy as np
from datetime import datetime, timedelta, time
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.utils import get_column_letter
import random
from statistics import stdev
# The 'drive' import is specific to Google Colab.
# If running locally, you might need to comment it out and adjust file paths.
from google.colab import drive
from tqdm import tqdm
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
import io
import os
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

def download_google_sheet_as_xlsx(spreadsheet_id, output_path="/content/scheduler_input.xlsx"):
    """
    Download Google Sheet by spreadsheet_id as .xlsx for pandas/openpyxl workflow.
    เหมาะกับโค้ดเดิมที่ใช้ pd.read_excel(sheet_name=...)
    """
    print("Authenticating Google account...")
    auth.authenticate_user()

    drive_service = build("drive", "v3")

    request = drive_service.files().export_media(
        fileId=spreadsheet_id,
        mimeType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    )

    with io.FileIO(output_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"Download progress: {int(status.progress() * 100)}%")

    print(f"Downloaded Google Sheet as Excel: {output_path}")
    return output_path


SCHEDULE_SOURCES = {
    "1": {
        "label": "เภสัชกร",
        "spreadsheet_id": "1yQi8CBW0wonXWqHs2vvSxq8rz-Th1EWo0piUmir8yss",
        "employee_sheet_name": "employee",
        "output_prefix": "Pharmacist_Schedule",
        "gsheet_prefix": "Pharmacist Schedule",
    },
    "2": {
        "label": "ผู้ช่วยเภสัชกร",
        "spreadsheet_id": "1SU4iOfeEPzqJmuQrNHtsDk1ItYe-lEWUCLA4gaG0BNc",
        "employee_sheet_name": "employee",
        "output_prefix": "Assistant_Pharmacist_Schedule",
        "gsheet_prefix": "Assistant Pharmacist Schedule",
    },
}


def ask_int_input(prompt_text, default_value, min_value=None, max_value=None):
    while True:
        raw_value = input(f"{prompt_text} [{default_value}]: ").strip()
        if raw_value == "":
            return default_value
        try:
            value = int(raw_value)
            if min_value is not None and value < min_value:
                print(f"กรุณาใส่ค่าตั้งแต่ {min_value} ขึ้นไป")
                continue
            if max_value is not None and value > max_value:
                print(f"กรุณาใส่ค่าไม่เกิน {max_value}")
                continue
            return value
        except ValueError:
            print("กรุณาใส่เป็นตัวเลขจำนวนเต็ม")




def ask_yes_no_input(prompt_text, default_value="N"):
    """
    Ask a Y/N question.
    Default is N unless explicitly changed.
    Returns True for Y and False for N.
    """
    default_value = str(default_value).strip().upper()
    if default_value not in ["Y", "N"]:
        default_value = "N"

    while True:
        raw_value = input(f"{prompt_text} [Y/N, default {default_value}]: ").strip().upper()
        if raw_value == "":
            raw_value = default_value

        if raw_value in ["Y", "YES"]:
            return True
        if raw_value in ["N", "NO"]:
            return False

        print("กรุณาตอบเฉพาะ Y หรือ N")


def ask_schedule_source():
    print("\nเลือกชนิดของการรันข้อมูล")
    print("1 = ตารางเภสัชกร")
    print("2 = ตารางผู้ช่วยเภสัชกร")
    while True:
        selected = input("กรุณาเลือก 1 หรือ 2 [1]: ").strip() or "1"
        if selected in SCHEDULE_SOURCES:
            return SCHEDULE_SOURCES[selected]
        print("กรุณาเลือกเฉพาะ 1 หรือ 2")


class PharmacistScheduler:
    """
    Pharmacy shift scheduler with optimization and Excel export.
    Designed for multi-constraint pharmacist roster planning.
    """
    W_CONSECUTIVE = 10
    W_HOURS = 4
    W_PREFERENCE = 6
    MAX_CONSECUTIVE_DAYS = 3
    MAX_HOUR_DIFF = 16

    # Soft constraints for smoother monthly distribution and weekend-off protection.
    # ปรับเป็นคะแนน ไม่ block แข็ง เพื่อไม่ให้เกิด UNFILLED โดยไม่จำเป็น
    MIN_WEEKEND_OFF_DAYS = 4
    W_SHIFT_PACING = 900
    W_MONTH_SEGMENT_BALANCE = 180
    W_WEEKEND_OFF_PROTECTION = 2200

    def __init__(self, excel_file_path, employee_sheet_name='employee', staff_type='เภสัชกร'):
        self.pharmacists = {}
        self.shift_types = {}
        self.departments = {}
        self.pre_assignments = {}
        self.historical_scores = {}
        self.preference_multipliers = {}
        self.special_notes = {}
        self.shift_limits = {}
        self.employee_sheet_name = employee_sheet_name
        self.staff_type = staff_type
        self.employee_order = []
        self.no_preference_staff = set()
        self.excel_file_path = excel_file_path
        self.problem_days = set()
        self.run_logs = []
        self.run_config = {}
        self.soul_mates = {
            'ภก.ชานนท์ (บุ้ง)': 'ภญ.อาภาภัทร (มะปราง)'


        }

        self.read_data_from_excel(self.excel_file_path)
        self.load_historical_scores()
        self._calculate_preference_multipliers()

        self.night_shifts = {
            'I100-10', 'I100-12N', 'I400-12N', 'I400-10', 'O400ER-12N', 'O400ER-10'
        }
        self.holidays = {
            'specific_dates': ['2026-07-28','2026-07-29','2026-07-30']  # ตัวอย่างวันหยุดเฉพาะวันที่
        }
        for pharmacist in self.pharmacists:
            self.pharmacists[pharmacist]['shift_counts'] = {
                shift_type: 0 for shift_type in self.shift_types
            }

    def _pre_check_staffing_levels(self, year, month):
        print("\nRunning pre-check for staffing levels (including all shifts + 3 buffer)...")
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) - timedelta(days=1) if month < 12 else datetime(year, 12, 31)
        dates = pd.date_range(start_date, end_date)

        all_ok = True
        for date in dates:
            available_pharmacists_count = sum(1 for p_name, p_info in self.pharmacists.items()
                                              if date.strftime('%Y-%m-%d') not in p_info['holidays'])
            required_shifts_base = sum(1 for st in self.shift_types
                                       if self.is_shift_available_on_date(st, date))
            total_required_shifts_with_buffer = required_shifts_base + 5

            if available_pharmacists_count < total_required_shifts_with_buffer:
                all_ok = False
                self.problem_days.add(date)
                print(f"WARNING: Potential shortage on {date.strftime('%Y-%m-%d')}. "
                      f"Available Pharmacists: {available_pharmacists_count}, "
                      f"Required Shifts (with +3 buffer): {total_required_shifts_with_buffer}")
        if all_ok:
            print("Pre-check complete. All days have sufficient staffing levels for the total workload.")
        else:
            print("Pre-check complete. Identified days with potential staff shortages. These will be prioritized.")
        return not all_ok


    def load_historical_scores(self):
        try:
            print("Attempting to load historical scores from sheet 'HistoricalScores'...")
            df = pd.read_excel(self.excel_file_path, sheet_name='HistoricalScores')
            if 'Pharmacist' in df.columns and 'Total Preference Score' in df.columns:
                for _, row in df.iterrows():
                    pharmacist = row['Pharmacist']
                    score = row['Total Preference Score']
                    if pharmacist in self.pharmacists:
                        self.historical_scores[pharmacist] = score
                print(f"Successfully loaded historical scores for {len(self.historical_scores)} pharmacists.")
            else:
                print("WARNING: 'HistoricalScores' sheet found, but required columns ('Pharmacist', 'Total Preference Score') are missing.")
        except ValueError:
            print("INFO: Sheet 'HistoricalScores' not found in the input file. Proceeding without historical data.")
        except Exception as e:
            print(f"An error occurred while loading historical scores: {e}")

    def _calculate_preference_multipliers(self):
        if not self.historical_scores:
            print("No historical scores found. All preference multipliers will be 1.0.")
            for pharmacist in self.pharmacists:
                self.preference_multipliers[pharmacist] = 1.0
            return
        min_score = min(self.historical_scores.values())
        max_score = max(self.historical_scores.values())
        if min_score == max_score:
            for pharmacist in self.pharmacists:
                self.preference_multipliers[pharmacist] = 1.0
            return
        for pharmacist, score in self.historical_scores.items():
            normalized_score = (score - min_score) / (max_score - min_score)
            min_multiplier = 0.7
            self.preference_multipliers[pharmacist] = min_multiplier + (1 - min_multiplier) * normalized_score
        for pharmacist in self.pharmacists:
            if pharmacist not in self.preference_multipliers:
                min_multiplier = 0.7
                self.preference_multipliers[pharmacist] = min_multiplier
                print(f"Pharmacist '{pharmacist}' not in historical data. Assigning a favorable multiplier of {min_multiplier}.")

    def _normalize_preference_value(self, value):
        if pd.isna(value):
            return None
        cleaned = str(value).strip()
        if cleaned == "" or cleaned.lower() == "none" or cleaned.lower() == "nan":
            return None
        return cleaned

    def _has_no_preferences(self, preferences):
        return all(self._normalize_preference_value(value) is None for value in preferences.values())

    def get_ordered_employees(self):
        ordered = [name for name in self.employee_order if name in self.pharmacists]
        missing = [name for name in self.pharmacists if name not in ordered]
        return ordered + missing

    def get_total_shift_count_in_schedule_dict(self, pharmacist, schedule_dict):
        count = 0
        for shifts in schedule_dict.values():
            for assigned in shifts.values():
                if assigned == pharmacist:
                    count += 1
        return count

    def get_unique_departments_worked(self, pharmacist, schedule_dict):
        departments = set()
        for shifts in schedule_dict.values():
            for shift_type, assigned in shifts.items():
                if assigned == pharmacist:
                    department = self.get_department_from_shift(shift_type)
                    if department:
                        departments.add(department)
        return departments

    def get_average_monthly_shift_target(self, year, month):
        """
        ประมาณจำนวนเวรเฉลี่ยต่อคนต่อเดือนจากจำนวนเวรที่เปิดจริงในเดือนนั้น
        ใช้เป็นฐาน pacing เพื่อไม่ให้คนเดิมถูกจัดหนักช่วงต้นเดือน/ปลายเดือน
        """
        cache_key = (year, month)
        if not hasattr(self, '_monthly_shift_target_cache'):
            self._monthly_shift_target_cache = {}
        if cache_key in self._monthly_shift_target_cache:
            return self._monthly_shift_target_cache[cache_key]

        start_date = datetime(year, month, 1)
        end_date = datetime(year + 1, 1, 1) - timedelta(days=1) if month == 12 else datetime(year, month + 1, 1) - timedelta(days=1)
        total_open_shifts = 0
        for date in pd.date_range(start_date, end_date):
            for shift_type in self.shift_types:
                if self.is_shift_available_on_date(shift_type, date):
                    total_open_shifts += 1

        staff_count = max(len(self.pharmacists), 1)
        target = max(total_open_shifts / staff_count, 1)
        self._monthly_shift_target_cache[cache_key] = target
        return target

    def _calculate_ideal_hours(self, year, month):
        """
        Target hours per employee for the month.
        Employees with a max_hours cap get at most min(cap, equal_share).
        Remainder distributes equally among uncapped employees.
        E.g. total=200, C(cap=60)->60, A->70, B->70
             total=300, C(cap=60)->60, A->120, B->120
        """
        cache_key = (year, month, 'ideal')
        if not hasattr(self, '_ideal_hours_cache'):
            self._ideal_hours_cache = {}
        if cache_key in self._ideal_hours_cache:
            return self._ideal_hours_cache[cache_key]

        DEFAULT_MAX = 250
        start_date = datetime(year, month, 1)
        end_date = (datetime(year + 1, 1, 1) - timedelta(days=1)
                    if month == 12 else datetime(year, month + 1, 1) - timedelta(days=1))
        total_hours = sum(
            self.shift_types[st]['hours']
            for date in pd.date_range(start_date, end_date)
            for st in self.shift_types
            if self.is_shift_available_on_date(st, date)
        )

        def get_cap(p):
            m = float(self.pharmacists[p].get('max_hours', DEFAULT_MAX))
            return m if m < DEFAULT_MAX else float('inf')

        sorted_pharm = sorted(self.pharmacists.keys(), key=get_cap)
        remaining, remaining_n = total_hours, len(sorted_pharm)
        ideal = {}
        for p in sorted_pharm:
            cap = get_cap(p)
            share = remaining / remaining_n if remaining_n > 0 else 0
            assigned = min(cap, share)
            ideal[p] = max(assigned, 0)
            remaining = max(remaining - assigned, 0)
            remaining_n -= 1

        self._ideal_hours_cache[cache_key] = ideal
        return ideal

    def get_month_segment(self, date):
        """แบ่งเดือนเป็น 3 ช่วง: ต้นเดือน / กลางเดือน / ปลายเดือน"""
        days_in_month = pd.Timestamp(date).days_in_month
        if date.day <= days_in_month / 3:
            return 'early'
        if date.day <= (2 * days_in_month) / 3:
            return 'middle'
        return 'late'

    def get_month_segment_shift_count(self, pharmacist, date, schedule_dict):
        """นับเวรของคนนี้ใน segment เดียวกับ date เพื่อกันการกระจุกในช่วงเดียวของเดือน"""
        target_segment = self.get_month_segment(date)
        count = 0
        for d, shifts in schedule_dict.items():
            if self.get_month_segment(d) != target_segment:
                continue
            if pharmacist in shifts.values():
                count += 1
        return count

    def get_total_weekend_days_in_schedule(self, schedule_dict):
        """นับจำนวนวันเสาร์-อาทิตย์ทั้งหมดในเดือนของ schedule_dict"""
        return sum(1 for d in schedule_dict if d.weekday() >= 5)

    def get_weekend_days_worked_until(self, pharmacist, schedule_dict):
        """นับจำนวนวันเสาร์-อาทิตย์ที่คนนี้ถูกจัดเวรแล้วใน schedule_dict ปัจจุบัน"""
        count = 0
        for d, shifts in schedule_dict.items():
            if d.weekday() >= 5 and pharmacist in shifts.values():
                count += 1
        return count

    def calculate_weekend_min_off_violations(self, schedule):
        """
        ตรวจสอบหลังจัดว่าใครมีวันหยุดเสาร์-อาทิตย์น้อยกว่า MIN_WEEKEND_OFF_DAYS
        คืนค่า dict: {pharmacist: shortfall}
        """
        weekend_dates = [d for d in schedule.index if d.weekday() >= 5]
        violations = {}
        for pharmacist in self.pharmacists:
            off_count = 0
            for d in weekend_dates:
                if pharmacist not in schedule.loc[d].values:
                    off_count += 1
            shortfall = max(0, self.MIN_WEEKEND_OFF_DAYS - off_count)
            if shortfall > 0:
                violations[pharmacist] = shortfall
        return violations

    def calculate_month_segment_variance(self, schedule):
        """
        วัดความกระจุกของเวรตามช่วงต้น/กลาง/ปลายเดือน
        ค่ายิ่งต่ำ = เวรของแต่ละคนกระจายตามเดือนดีกว่า
        """
        variances = []
        for pharmacist in self.pharmacists:
            counts = {'early': 0, 'middle': 0, 'late': 0}
            for d in schedule.index:
                if pharmacist in schedule.loc[d].values:
                    counts[self.get_month_segment(d)] += 1
            variances.append(np.var(list(counts.values())))
        return float(np.mean(variances)) if variances else 0

    def read_data_from_excel(self, file_path):
        # 1. โหลดข้อมูล Skill Group จากชีต 'Skill subset'
        skill_groups_map = {}
        try:
            print("Attempting to load skill subsets from sheet 'Skill subset'...")
            subset_df = pd.read_excel(file_path, sheet_name='Skill subset')

            # ตรวจสอบว่ามีคอลัมน์ที่ต้องการหรือไม่
            if 'Group Name' in subset_df.columns and 'Skills' in subset_df.columns:
                for _, row in subset_df.iterrows():
                    group_name = str(row['Group Name']).strip()
                    if group_name and group_name != 'nan':
                        # แยก Skill ด้วย comma และลบช่องว่างหน้าหลัง
                        skills_list = [s.strip() for s in str(row['Skills']).split(',') if s.strip() and s.strip() != 'nan']
                        skill_groups_map[group_name] = skills_list
                print(f"Successfully loaded {len(skill_groups_map)} skill groups.")
            else:
                print("WARNING: 'Skill subset' sheet found, but required columns ('Group Name', 'Skills') are missing.")
        except ValueError:
            print("INFO: Sheet 'Skill subset' not found. Proceeding without predefined skill groups.")
        except Exception as e:
            print(f"An error occurred while loading skill subsets: {e}")

        # 2. อ่านข้อมูลเจ้าหน้าที่จากชีต employee และ Map Skills
        pharmacists_df = pd.read_excel(file_path, sheet_name=self.employee_sheet_name)
        self.pharmacists = {}
        self.employee_order = []
        self.no_preference_staff = set()
        for _, row in pharmacists_df.iterrows():
            name = str(row['Name']).strip()
            if not name or name.lower() == 'nan':
                continue
            self.employee_order.append(name)
            max_hours = row.get('Max Hours', 250)
            if pd.isna(max_hours) or max_hours == '' or max_hours is None:
                max_hours = 250
            else:
                max_hours = float(max_hours)

            # ลอจิกการแตก Skill Group เป็น Skill ย่อย
            raw_skills = str(row['Skills']).split(',')
            expanded_skills = set()
            for s in raw_skills:
                s_clean = s.strip()
                # ถ้าเจอคำที่ตรงกับ Group Name ในชีต Skill subset ให้แตกเป็น List
                if s_clean in skill_groups_map:
                    expanded_skills.update(skill_groups_map[s_clean])
                elif s_clean and s_clean != 'nan':
                    expanded_skills.add(s_clean)

            preferences = {
                f'rank{i}': self._normalize_preference_value(row.get(f'Rank{i}'))
                for i in range(1, 9)
            }
            no_preference = self._has_no_preferences(preferences)
            if no_preference:
                self.no_preference_staff.add(name)

            self.pharmacists[name] = {
                'night_shift_count': 0,
                'skills': list(expanded_skills),
                'holidays': [date for date in str(row.get('Holidays', '')).split(',') if date != '1900-01-00' and date.strip() and date != 'nan'],
                'shift_counts': {},
                'preferences': preferences,
                'no_preference': no_preference,
                'max_hours': max_hours
            }

        # ... (โค้ดส่วนที่เหลือของฟังก์ชันตั้งแต่การอ่านชีต 'Shifts' ยังคงเหมือนเดิม) ...
        shifts_df = pd.read_excel(file_path, sheet_name='Shifts')
        # ...
        self.shift_types = {}
        for _, row in shifts_df.iterrows():
            shift_code = row['Shift Code']
            if pd.isna(shift_code) or str(shift_code).strip() in ('', 'nan', 'NaT'):
                continue
            shift_code = str(shift_code).strip()
            self.shift_types[shift_code] = {
                'description': row['Description'],
                'shift_type': row['Shift Type'],
                'start_time': row['Start Time'],
                'end_time': row['End Time'],
                'hours': (
                    (row['Hours'].hour + row['Hours'].minute / 60)
                    if hasattr(row['Hours'], 'hour')
                    else (
                        int(str(row['Hours']).split(':')[0]) + int(str(row['Hours']).split(':')[1]) / 60
                        if ':' in str(row['Hours'])
                        else float(str(row['Hours']))
                    )
                ),
                'required_skills': str(row['Required Skills']).split(','),
                'restricted_next_shifts': str(row['Restricted Next Shifts']).split(',') if pd.notna(row['Restricted Next Shifts']) else [],
            }
        departments_df = pd.read_excel(file_path, sheet_name='Departments')
        self.departments = {}
        for _, row in departments_df.iterrows():
            department = row['Department']
            if pd.isna(department) or str(department).strip() in ('', 'nan', 'NaT'):
                continue
            self.departments[department] = str(row['Shift Codes']).split(',')
        pre_assign_df = pd.read_excel(file_path, sheet_name='PreAssignments')
        pre_assign_df['Date'] = pd.to_datetime(pre_assign_df['Date']).dt.strftime('%Y-%m-%d')
        self.pre_assignments = {}
        for pharmacist, group in pre_assign_df.groupby('Pharmacist'):
            date_dict = {}
            for date, g in group.groupby('Date'):
                shifts = []
                for shift_str in g['Shift']:
                    shifts.extend([s.strip() for s in str(shift_str).split(',') if s.strip()])
                date_dict[date] = shifts
            self.pre_assignments[pharmacist] = date_dict

        try:
            print("Attempting to load special notes from sheet 'SpecialNotes'...")
            notes_df = pd.read_excel(file_path, sheet_name='SpecialNotes', index_col=0)
            for pharmacist, row_data in notes_df.iterrows():
                if pharmacist in self.pharmacists:
                    for date_col, note in row_data.items():
                        if pd.notna(note) and str(note).strip():
                            date_str = pd.to_datetime(date_col).strftime('%Y-%m-%d')
                            if pharmacist not in self.special_notes:
                                self.special_notes[pharmacist] = {}
                            self.special_notes[pharmacist][date_str] = str(note).strip()
            print(f"Successfully loaded {sum(len(d) for d in self.special_notes.values())} special notes.")
        except ValueError:
            print("INFO: Sheet 'SpecialNotes' not found. Proceeding without special notes.")
        except Exception as e:
            print(f"An error occurred while loading special notes: {e}")

        try:
            print("Attempting to load shift limits from sheet 'ShiftLimits'...")
            limits_df = pd.read_excel(file_path, sheet_name='ShiftLimits')
            for _, row in limits_df.iterrows():
                pharmacist = row['Pharmacist']
                category = row['ShiftCategory']
                max_count = row['MaxCount']
                if pharmacist in self.pharmacists:
                    if pharmacist not in self.shift_limits:
                        self.shift_limits[pharmacist] = {}
                    self.shift_limits[pharmacist][category] = int(max_count)
            print(f"Successfully loaded {len(limits_df)} shift limit rules.")
        except ValueError:
            print("INFO: Sheet 'ShiftLimits' not found. Proceeding without shift limits.")
        except Exception as e:
            print(f"An error occurred while loading shift limits: {e}")
        # --- MIN SHIFT REQUIREMENTS ---
        self.min_shift_requirements = {}  # {pharmacist: {department: min_count}}
        try:
            print("Attempting to load minimum shift requirements from sheet 'MinShiftRequirements'...")
            min_req_df = pd.read_excel(file_path, sheet_name='MinShiftRequirements')
            for _, row in min_req_df.iterrows():
                pharmacist = str(row['Pharmacist']).strip()
                department = str(row['Department']).strip()
                min_count  = int(float(str(row['MinCount']))) if pd.notna(row['MinCount']) else 0
                if pharmacist in self.pharmacists:
                    if pharmacist not in self.min_shift_requirements:
                        self.min_shift_requirements[pharmacist] = {}
                    self.min_shift_requirements[pharmacist][department] = min_count
            print(f"Successfully loaded min shift requirements for {len(self.min_shift_requirements)} pharmacists.")
        except ValueError:
            print("INFO: Sheet 'MinShiftRequirements' not found. Proceeding without min shift requirements.")
            self.min_shift_requirements = {}
        except Exception as e:
            print(f"An error occurred while loading min shift requirements: {e}")
            self.min_shift_requirements = {}

    def convert_time_to_minutes(self, time_input):
        if isinstance(time_input, str):
            hours, minutes = map(int, time_input.split(':'))
        elif isinstance(time_input, time):
            hours, minutes = time_input.hour, time_input.minute
        else:
            raise ValueError("Invalid input type. Expected string (HH:MM) or datetime.time object.")
        return hours * 60 + minutes

    def check_time_overlap(self, start1, end1, start2, end2):
        start1_mins = self.convert_time_to_minutes(start1)
        end1_mins = self.convert_time_to_minutes(end1)
        start2_mins = self.convert_time_to_minutes(start2)
        end2_mins = self.convert_time_to_minutes(end2)
        if end1_mins < start1_mins: end1_mins += 24 * 60
        if end2_mins < start2_mins: end2_mins += 24 * 60
        return start1_mins < end2_mins and end1_mins > start2_mins

    def check_mixing_expert_ratio_optimized(self, schedule_dict, date, current_shift=None, current_pharm=None):
        mixing_shifts = [p for s, p in schedule_dict[date].items()
                         if s.startswith('C8') and p not in ['NO SHIFT', 'UNASSIGNED', 'UNFILLED']]
        if current_shift and current_shift.startswith('C8') and current_pharm:
            mixing_shifts.append(current_pharm)
        if not mixing_shifts: return True
        total_mixing = len(mixing_shifts)
        expert_count = sum(1 for pharm in mixing_shifts
                           if pharm in self.pharmacists and 'mixing_expert' in self.pharmacists[pharm]['skills'])
        return expert_count >= (2 * total_mixing / 3)

    def count_consecutive_shifts(self, pharmacist, date, schedule, max_days=6):
        count = 0
        current_date = date - timedelta(days=1)
        for _ in range(max_days):
            if current_date in schedule.index and pharmacist in schedule.loc[current_date].values:
                count += 1
                current_date -= timedelta(days=1)
            else:
                break
        return count

    def is_holiday(self, date):
        return date.strftime('%Y-%m-%d') in self.holidays['specific_dates']

    def get_dept_shift_count(self, pharmacist, department, schedule_dict):
        """
        นับจำนวนเวรที่ pharmacist ทำในแผนก department แล้วใน schedule_dict ปัจจุบัน
        """
        count = 0
        for date, shifts in schedule_dict.items():
            for shift_type, assigned in shifts.items():
                if assigned == pharmacist:
                    if self.get_department_from_shift(shift_type) == department:
                        count += 1
        return count


    def _needs_min_shift(self, pharmacist, shift_type, schedule_dict):
        """
        คืนค่า True ถ้า pharmacist ยังไม่ถึง MinCount ของ department นี้
        ใช้เป็น priority boost ใน scoring
        """
        dept = self.get_department_from_shift(shift_type)
        if not dept:
            return False
        min_req = self.min_shift_requirements.get(pharmacist, {}).get(dept, 0)
        if min_req == 0:
            return False
        current_count = self.get_dept_shift_count(pharmacist, dept, schedule_dict)
        return current_count < min_req


    def calculate_weekend_off_variance(self, schedule, year, month):
        weekend_off_counts = {p: 0 for p in self.pharmacists}
        start_date = datetime(year, month, 1)
        end_date = datetime(year, month + 1, 1) - timedelta(days=1) if month < 12 else datetime(year, 12, 31)
        for date in pd.date_range(start_date, end_date):
            if date.weekday() >= 5:
                working_on_weekend = {schedule.loc[date, shift] for shift in schedule.columns if schedule.loc[date, shift] not in ['NO SHIFT', 'UNFILLED', 'UNASSIGNED']}
                for p_name in self.pharmacists:
                    if p_name not in working_on_weekend:
                        weekend_off_counts[p_name] += 1
        if len(weekend_off_counts) > 1:
            return np.var(list(weekend_off_counts.values()))
        return 0

    def is_night_shift(self, shift_type):
        if shift_type in self.night_shifts:
            return True
        if shift_type in self.shift_types:
            s_type = str(self.shift_types[shift_type].get('shift_type', '')).strip().lower()
            if s_type.endswith('n') and any(c.isdigit() for c in s_type):
                return True
        return False

    def _get_holiday_blocks(self, year, month):
        """
        คำนวณ block วันหยุดยาวทั้งหมดในเดือน
        คืนค่า dict: {last_day (datetime): set of all days in block}
        """
        import calendar
        start_date = datetime(year, month, 1)
        if month == 12:
            end_date = datetime(year, 12, 31)
        else:
            end_date = datetime(year, month + 1, 1) - timedelta(days=1)

        specific_holidays = set(
            datetime.strptime(d, '%Y-%m-%d')
            for d in self.holidays['specific_dates']
        )

        # สร้าง set วันที่เป็น "วันหยุด" ทุกประเภท (เสาร์, อาทิตย์, specific)
        all_off_days = set()
        current = start_date - timedelta(days=7)  # buffer ก่อนเดือน
        scan_end = end_date + timedelta(days=7)    # buffer หลังเดือน
        d = current
        while d <= scan_end:
            if d.weekday() >= 5 or d in specific_holidays:
                all_off_days.add(d)
            d += timedelta(days=1)

        # จัดกลุ่มเป็น block ติดกัน
        sorted_days = sorted(all_off_days)
        blocks = []
        if not sorted_days:
            return {}

        current_block = [sorted_days[0]]
        for day in sorted_days[1:]:
            if (day - current_block[-1]).days == 1:
                current_block.append(day)
            else:
                blocks.append(current_block)
                current_block = [day]
        blocks.append(current_block)

        # คืนค่า {last_day: set(block)}
        result = {}
        for block in blocks:
            # block ต้องมีเสาร์หรืออาทิตย์อยู่ด้วย (ไม่ใช่แค่ specific holiday กลางสัปดาห์โดด ๆ)
            has_weekend = any(d.weekday() >= 5 for d in block)
            if has_weekend and len(block) >= 1:
                last_day = block[-1]
                result[last_day] = set(block)

        return result


    def is_shift_available_on_date(self, shift_type, date):
        shift_info = self.shift_types[shift_type]
        is_holiday_date = self.is_holiday(date)  # specific_dates เท่านั้น

        weekday_num = date.weekday()
        is_saturday = (weekday_num == 5)
        is_sunday   = (weekday_num == 6)
        is_mon_to_thu = (0 <= weekday_num <= 3)

        s_type = str(shift_info['shift_type']).strip().lower()

        # --- Hour-based values (e.g. '8', '10N', '8*') from the Shifts sheet ---
        # Night suffix: '10n', '12n' -> available every day
        if s_type.endswith('n') and any(c.isdigit() for c in s_type):
            return True
        # Asterisk suffix: '8*', '10*' -> weekend / holiday shifts
        if s_type.endswith('*'):
            return is_saturday or is_sunday or is_holiday_date
        # Plain hour number: '8', '6', '4', '12', '10', etc. -> weekday only
        if s_type.replace('.', '').isdigit():
            return not (is_holiday_date or is_saturday or is_sunday)

        # --- Keyword-based values (kept for backward compatibility) ---
        if s_type in ['weekday', 'จันทร์ - ศุกร์']:
            return not (is_holiday_date or is_saturday or is_sunday)
        elif s_type in ['saturday', 'เสาร์']:
            return is_saturday and not is_holiday_date
        elif s_type in ['sat-holiday', 'เสาร์, หยุดฯ']:
            return is_saturday or is_holiday_date
        elif s_type == 'sunday':
            return is_sunday and not is_holiday_date
        elif s_type in ['weekend', 'เสาร์ - อาทิตย์, หยุดฯ']:
            return is_saturday or is_sunday or is_holiday_date
        elif s_type == 'sun-holiday':
            return is_sunday and is_holiday_date
        elif s_type in ['mon-thu', 'วันจันทร์-พฤหัส', 'วันจันทร์-พฤหัสบดี', 'จันทร์-พฤหัส']:
            return is_mon_to_thu and not is_holiday_date
        elif s_type == 'holiday':
            return is_holiday_date and not is_saturday and not is_sunday
        elif s_type in ['last-day-holiday', 'อาทิตย์, หยุดสุดท้ายของเทศกาล', 'อา,หยุดสุดท้ายของวันหยุดยาว']:
            year  = date.year
            month = date.month
            holiday_blocks = self._get_holiday_blocks(year, month)
            return date in holiday_blocks
        elif s_type == 'night':
            return True

        return False

    def get_department_from_shift(self, shift_type):
        if shift_type.startswith('I100'): return 'IPD100'
        elif shift_type.startswith('O100'): return 'OPD100'
        elif shift_type.startswith('Care'): return 'Care'
        elif shift_type.startswith('C8'): return 'Mixing'
        elif shift_type.startswith('I400'): return 'IPD400'
        elif shift_type.startswith('O400F1'): return 'OPD400F1'
        elif shift_type.startswith('O400F2'): return 'OPD400F2'
        elif shift_type.startswith('O400ER'): return 'ER'
        elif shift_type.startswith('ARI'): return 'ARI'
        elif shift_type.startswith('Refill'): return 'Refill'
        return None

    def _get_shift_category(self, shift_type):
        if self.is_night_shift(shift_type):
            return 'Night'
        if shift_type.startswith('C8'):
            return 'Mixing'
        return None

    def get_night_shift_count(self, pharmacist):
        return self.pharmacists[pharmacist]['night_shift_count']

    def get_preference_score(self, pharmacist, shift_type):
        p_skills = [skill.strip().lower() for skill in self.pharmacists[pharmacist]['skills']]
        if 'junior' in p_skills:
            return 1  # ให้คะแนนความชอบเป็น 1 (ดีที่สุด) เสมอ เพื่อให้กระจายไปได้ทุกที่

        # คนที่ไม่ได้ระบุ Preference เลย หรือระบุเป็น none/None ทุกช่อง
        # ให้ใช้คะแนนกลาง ไม่กดไปท้ายคิว และให้ logic department balance เป็นตัวกระจายงานแทน
        if self.pharmacists[pharmacist].get('no_preference', False):
            return 5

        department = self.get_department_from_shift(shift_type)
        for rank in range(1, 9):
            if self.pharmacists[pharmacist]['preferences'].get(f'rank{rank}') == department:
                return rank
        return 9

    def has_restricted_sequence_optimized(self, pharmacist, date, shift_type, schedule_dict):
        previous_date = date - timedelta(days=1)
        if previous_date in schedule_dict:
            for prev_shift, assigned_pharm in schedule_dict[previous_date].items():
                if assigned_pharm == pharmacist:
                    restricted = self.shift_types[prev_shift].get('restricted_next_shifts', [])
                    if shift_type in restricted: return True
        return False

    def has_overlapping_shift_optimized(self, pharmacist, date, new_shift_type, schedule_dict):
        if date not in schedule_dict: return False
        new_start = self.shift_types[new_shift_type]['start_time']
        new_end = self.shift_types[new_shift_type]['end_time']
        for existing_shift, assigned_pharm in schedule_dict[date].items():
            if assigned_pharm == pharmacist and existing_shift != new_shift_type:
                existing_start = self.shift_types[existing_shift]['start_time']
                existing_end = self.shift_types[existing_shift]['end_time']
                if self.check_time_overlap(new_start, new_end, existing_start, existing_end):
                    return True
        return False

    def has_nearby_night_shift_optimized(self, pharmacist, date, schedule_dict):
        for delta in [-2, -1, 1, 2]:
            check_date = date + timedelta(days=delta)
            if check_date in schedule_dict:
                for shift, assigned_pharm in schedule_dict[check_date].items():
                    if assigned_pharm == pharmacist and self.is_night_shift(shift):
                        return True
        return False

    def get_pharmacist_shifts(self, pharmacist, date, current_schedule):
        shifts = []
        if date in current_schedule.index:
            for shift_type in current_schedule.columns:
                if current_schedule.loc[date, shift_type] == pharmacist:
                    shifts.append(shift_type)
        return shifts

    def calculate_total_hours(self, pharmacist, schedule):
        total_hours = 0
        for date in schedule.index:
            for shift_type, assigned_pharm in schedule.loc[date].items():
                if assigned_pharm == pharmacist and shift_type in self.shift_types:
                    total_hours += self.shift_types[shift_type]['hours']
        return total_hours

    def _get_hour_imbalance_penalty(self, hours_dict):
        if not hours_dict or len(hours_dict) < 2:
            return 0
        hour_values = list(hours_dict.values())
        hour_stdev = stdev(hour_values)
        hour_range = max(hour_values) - min(hour_values)
        stdev_penalty = hour_stdev ** 2
        range_penalty = 0
        if hour_range > 10:
            range_penalty = (hour_range - 10) ** 2
        return stdev_penalty + range_penalty

    def calculate_schedule_metrics(self, schedule, year, month):
        hours = {p: self.calculate_total_hours(p, schedule) for p in self.pharmacists}
        night_counts = {p: self.pharmacists[p]['night_shift_count'] for p in self.pharmacists}
        weekend_off_var = self.calculate_weekend_off_variance(schedule, year, month)
        hour_penalty = self._get_hour_imbalance_penalty(hours)

        # --- ADDED: คำนวณความแปรปรวนของ Preference Score (%) ---
        pref_percentages = self.calculate_pharmacist_preference_scores(schedule)
        pref_variance = np.var(list(pref_percentages.values())) if len(pref_percentages) > 1 else 0

        weekend_min_off_violations = self.calculate_weekend_min_off_violations(schedule)
        metrics = {
            'hour_imbalance_penalty': hour_penalty,
            'night_variance': np.var(list(night_counts.values())) if night_counts else 0,
            'preference_score': sum(self.calculate_preference_penalty(p, schedule) for p in self.pharmacists),
            'preference_variance': pref_variance,
            'weekend_off_variance': weekend_off_var,
            'weekend_min_off_shortfall': sum(weekend_min_off_violations.values()),
            'month_segment_variance': self.calculate_month_segment_variance(schedule),
        }
        if len(hours) > 1:
            metrics['hour_diff_for_logging'] = stdev(hours.values())
        else:
            metrics['hour_diff_for_logging'] = 0
        return metrics

    def _log_schedule_event(self, event_type, message, **kwargs):
        """
        Store runtime logs in memory.
        These logs are exported only when enable_run_log=True.
        """
        log_row = {
            "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Event Type": event_type,
            "Message": message,
        }
        for key, value in kwargs.items():
            log_row[key] = value
        self.run_logs.append(log_row)

    def _reset_runtime_shift_counters(self):
        """
        Reset counters used during schedule generation.
        """
        for pharmacist in self.pharmacists:
            self.pharmacists[pharmacist]['night_shift_count'] = 0
            self.pharmacists[pharmacist]['mixing_shift_count'] = 0
            self.pharmacists[pharmacist]['care_shift_count'] = 0
            self.pharmacists[pharmacist]['category_counts'] = {
                'Mixing': 0,
                'Night': 0
            }

    def _has_required_skills_for_shift(self, pharmacist, shift_type):
        """
        True when pharmacist has all required skills for shift_type.
        Empty Required Skills means everyone can be assigned.
        """
        p_skills = {
            str(skill).strip().lower()
            for skill in self.pharmacists.get(pharmacist, {}).get('skills', [])
            if str(skill).strip() and str(skill).strip().lower() != 'nan'
        }
        required_skills = {
            str(skill).strip().lower()
            for skill in self.shift_types.get(shift_type, {}).get('required_skills', [])
            if str(skill).strip() and str(skill).strip().lower() != 'nan'
        }
        return required_skills.issubset(p_skills)

    def _get_true_random_candidates(self, staff_pool, date, shift_type, schedule_dict):
        """
        Candidate filter for TRUE RANDOM OVERRIDE mode.

        This mode ignores scoring/fairness/preference/limits, but still keeps:
        1) no more than one shift per staff per day
        2) no assignment on personal holiday/leave
        3) required skill matching
        """
        date_str = pd.to_datetime(date).strftime('%Y-%m-%d')
        candidates = []

        for pharmacist in staff_pool:
            if pharmacist not in self.pharmacists:
                continue

            # 1) Do not assign multiple shifts in the same day.
            if pharmacist in schedule_dict[date].values():
                continue

            # 2) Do not assign staff on holiday/leave.
            if date_str in self.pharmacists[pharmacist].get('holidays', []):
                continue

            # 3) Required skill must match.
            if not self._has_required_skills_for_shift(pharmacist, shift_type):
                continue

            candidates.append(pharmacist)

        return candidates

    def _select_fair_random_candidate(self, candidates, pharmacist_hours, fairness_buffer_hours=8):
        """
        Select candidate using fairness-aware random logic.

        Logic:
        - Keep randomness, but do not random from the whole candidate pool.
        - First restrict pool to staff whose current total hours are close to the lowest current hours.
        - Then random inside that fair pool.

        fairness_buffer_hours=8 means:
        คนที่ชั่วโมงปัจจุบันไม่เกินคนชั่วโมงน้อยสุด + 8 ชั่วโมง จะมีสิทธิ์ถูกสุ่ม
        """
        if not candidates:
            return None, []

        min_hours = min(pharmacist_hours.get(p, 0) for p in candidates)
        fair_pool = [
            p for p in candidates
            if pharmacist_hours.get(p, 0) <= min_hours + fairness_buffer_hours
        ]

        if not fair_pool:
            fair_pool = candidates

        return random.choice(fair_pool), fair_pool

    def generate_monthly_schedule_true_random(self, year, month, iteration_num=1):
        """
        TRUE RANDOM OVERRIDE MODE - constrained + fairness-aware random.

        Randomly assigns open shifts while ignoring:
        - preference score
        - consecutive day limit
        - night spacing
        - restricted next shift rule
        - junior pairing rule
        - mixing ratio rule
        - shift category limits
        - historical score/multiplier
        - rescue/swap scoring

        Still enforced:
        - shift must be open on that date
        - PreAssignments are applied first and never overwritten
        - one staff cannot be assigned more than one shift in the same day
        - staff on holiday/leave cannot be assigned
        - staff must have all Required Skills for the shift
        - fairness / hour balance is preserved by randomizing only among lower-hour candidates
        """
        self._reset_runtime_shift_counters()

        start_date = datetime(year, month, 1)
        end_date = (
            datetime(year + 1, 1, 1) - timedelta(days=1)
            if month == 12
            else datetime(year, month + 1, 1) - timedelta(days=1)
        )
        dates = pd.date_range(start_date, end_date)

        schedule_dict = {
            date: {shift: 'NO SHIFT' for shift in self.shift_types}
            for date in dates
        }
        pharmacist_hours = {p: 0 for p in self.pharmacists}
        staff_pool = list(self.pharmacists.keys())

        unfilled_info = {
            'problem_days': [],
            'other_days': []
        }

        self._log_schedule_event(
            "TRUE_RANDOM_START",
            "Start constrained true random schedule generation",
            Year=year,
            Month=month,
            Iteration=iteration_num,
            StaffCount=len(staff_pool)
        )

        # 1) Apply locked PreAssignments first.
        for pharmacist, assignments in self.pre_assignments.items():
            if pharmacist not in self.pharmacists:
                self._log_schedule_event(
                    "PREASSIGN_SKIPPED",
                    "Preassigned staff not found in employee list",
                    Pharmacist=pharmacist
                )
                continue

            for date_str, shift_types in assignments.items():
                date = pd.to_datetime(date_str)
                if date not in schedule_dict:
                    continue

                for shift_type in shift_types:
                    if shift_type not in self.shift_types:
                        self._log_schedule_event(
                            "PREASSIGN_SKIPPED",
                            "Preassigned shift code not found in Shifts sheet",
                            Date=date.strftime("%Y-%m-%d"),
                            Shift=shift_type,
                            Pharmacist=pharmacist
                        )
                        continue

                    # Do not block PreAssignments even if they violate leave/skill/day rules.
                    # They are treated as manual locked assignments.
                    schedule_dict[date][shift_type] = pharmacist
                    self._update_shift_counts(pharmacist, shift_type)
                    pharmacist_hours[pharmacist] += self.shift_types[shift_type]['hours']

                    self._log_schedule_event(
                        "PREASSIGN_APPLIED",
                        "Applied locked preassignment",
                        Date=date.strftime("%Y-%m-%d"),
                        Shift=shift_type,
                        Pharmacist=pharmacist
                    )

        # 2) Randomly assign all open shifts with only hard safety filters.
        for date in tqdm(dates, desc=f"True Random Schedule (Iteration {iteration_num})", leave=False):
            shifts_to_process = list(self.shift_types.keys())
            random.shuffle(shifts_to_process)

            for shift_type in shifts_to_process:
                # Keep closed shifts as NO SHIFT.
                if not self.is_shift_available_on_date(shift_type, date):
                    schedule_dict[date][shift_type] = 'NO SHIFT'
                    continue

                # Do not overwrite PreAssignments or any existing assignment.
                if schedule_dict[date][shift_type] not in ['NO SHIFT', 'UNASSIGNED', 'UNFILLED']:
                    continue

                candidates = self._get_true_random_candidates(
                    staff_pool=staff_pool,
                    date=date,
                    shift_type=shift_type,
                    schedule_dict=schedule_dict
                )

                if not candidates:
                    schedule_dict[date][shift_type] = 'UNFILLED'
                    if date in self.problem_days:
                        unfilled_info['problem_days'].append((date, shift_type))
                    else:
                        unfilled_info['other_days'].append((date, shift_type))

                    self._log_schedule_event(
                        "UNFILLED_TRUE_RANDOM",
                        "No candidate after applying hard filters: one shift/day, not on leave, required skills",
                        Date=date.strftime("%Y-%m-%d"),
                        Shift=shift_type
                    )
                    continue

                chosen, fair_pool = self._select_fair_random_candidate(
                    candidates=candidates,
                    pharmacist_hours=pharmacist_hours,
                    fairness_buffer_hours=8
                )

                if chosen is None:
                    schedule_dict[date][shift_type] = 'UNFILLED'
                    if date in self.problem_days:
                        unfilled_info['problem_days'].append((date, shift_type))
                    else:
                        unfilled_info['other_days'].append((date, shift_type))
                    self._log_schedule_event(
                        "UNFILLED_TRUE_RANDOM",
                        "No chosen candidate after fairness filter",
                        Date=date.strftime("%Y-%m-%d"),
                        Shift=shift_type
                    )
                    continue

                before_hours = pharmacist_hours.get(chosen, 0)
                schedule_dict[date][shift_type] = chosen
                self._update_shift_counts(chosen, shift_type)
                pharmacist_hours[chosen] += self.shift_types[shift_type]['hours']

                self._log_schedule_event(
                    "TRUE_RANDOM_ASSIGN",
                    "Assigned by fairness-aware constrained true random override",
                    Date=date.strftime("%Y-%m-%d"),
                    Shift=shift_type,
                    Pharmacist=chosen,
                    CandidateCount=len(candidates),
                    FairPoolCount=len(fair_pool),
                    HoursBefore=before_hours,
                    ShiftHours=self.shift_types[shift_type]['hours'],
                    HoursAfter=pharmacist_hours[chosen]
                )

        self._fix_post_night_violations(schedule_dict, pharmacist_hours, {p: 0 for p in self.pharmacists}, staff_pool)
        final_schedule = pd.DataFrame.from_dict(schedule_dict, orient='index')
        final_schedule = final_schedule.reindex(columns=list(self.shift_types.keys()), fill_value='NO SHIFT')
        final_schedule.fillna('NO SHIFT', inplace=True)

        self._log_schedule_event(
            "TRUE_RANDOM_COMPLETE",
            "Completed constrained true random schedule generation",
            Year=year,
            Month=month,
            TotalUnfilled=len(unfilled_info['problem_days']) + len(unfilled_info['other_days'])
        )

        return final_schedule, unfilled_info

    def generate_monthly_schedule_shuffled(self, year, month, shuffled_shifts=None, shuffled_pharmacists=None, iteration_num=1):
        start_date = datetime(year, month, 1)
        end_date = datetime(year + 1, 1, 1) - timedelta(days=1) if month == 12 else datetime(year, month + 1, 1) - timedelta(days=1)
        dates = pd.date_range(start_date, end_date)
        schedule_dict = {date: {shift: 'NO SHIFT' for shift in self.shift_types} for date in dates}
        pharmacist_hours = {p: 0 for p in self.pharmacists}
        pharmacist_consecutive_days = {p: 0 for p in self.pharmacists}

        if shuffled_shifts is None:
            shuffled_shifts = list(self.shift_types.keys())
            random.shuffle(shuffled_shifts)
        if shuffled_pharmacists is None:
            shuffled_pharmacists = list(self.pharmacists.keys())
            random.shuffle(shuffled_pharmacists)

        for pharmacist in self.pharmacists:
            self.pharmacists[pharmacist]['night_shift_count'] = 0
            self.pharmacists[pharmacist]['mixing_shift_count'] = 0
            self.pharmacists[pharmacist]['care_shift_count'] = 0
            self.pharmacists[pharmacist]['category_counts'] = {
                'Mixing': 0,
                'Night': 0
            }

        for pharmacist, assignments in self.pre_assignments.items():
            if pharmacist not in self.pharmacists: continue
            for date_str, shift_types in assignments.items():
                date = pd.to_datetime(date_str)
                if date not in schedule_dict: continue
                for shift_type in shift_types:
                    if shift_type in self.shift_types:
                        schedule_dict[date][shift_type] = pharmacist
                        self._update_shift_counts(pharmacist, shift_type)
                        pharmacist_hours[pharmacist] += self.shift_types[shift_type]['hours']

        all_dates = list(pd.date_range(start_date, end_date))
        problem_dates_shuffled = [d for d in all_dates if d in self.problem_days]
        random.shuffle(problem_dates_shuffled)
        other_dates_shuffled = [d for d in all_dates if d not in self.problem_days]
        random.shuffle(other_dates_shuffled)
        processing_order_dates = problem_dates_shuffled + other_dates_shuffled
        unfilled_info = {'problem_days': [], 'other_days': []}
        night_shifts_ordered = [s for s in shuffled_shifts if self.is_night_shift(s)]
        mixing_shifts_ordered = [s for s in shuffled_shifts if s.startswith('C8') and not self.is_night_shift(s)]
        care_shifts_ordered = [s for s in shuffled_shifts if s.startswith('Care') and not self.is_night_shift(s) and not s.startswith('C8')]
        other_shifts_ordered = [s for s in shuffled_shifts if not self.is_night_shift(s) and not s.startswith('C8') and not s.startswith('Care')]
        standard_shift_order = night_shifts_ordered + mixing_shifts_ordered + care_shifts_ordered + other_shifts_ordered
        problem_day_shift_order = mixing_shifts_ordered + care_shifts_ordered + night_shifts_ordered + other_shifts_ordered

        for date in tqdm(processing_order_dates, desc=f"Building Schedule (Iteration {iteration_num})", leave=False):
            pharmacists_working_yesterday = set()
            previous_date = date - timedelta(days=1)
            if previous_date in schedule_dict:
                   pharmacists_working_yesterday = {p for p in schedule_dict[previous_date].values() if p in self.pharmacists}
            for p_name in self.pharmacists:
                if p_name in pharmacists_working_yesterday:
                    pharmacist_consecutive_days[p_name] += 1
                else:
                    pharmacist_consecutive_days[p_name] = 0

            is_day_before_problem_day = (date + timedelta(days=1)) in self.problem_days

            shifts_to_process = problem_day_shift_order if date in self.problem_days else standard_shift_order
            for shift_type in shifts_to_process:
                if schedule_dict[date][shift_type] not in ['NO SHIFT', 'UNASSIGNED', 'UNFILLED'] or not self.is_shift_available_on_date(shift_type, date):
                    continue
                available = self._get_available_pharmacists_optimized(shuffled_pharmacists, date, shift_type, schedule_dict, pharmacist_hours, pharmacist_consecutive_days)
                if available:
                    chosen = self._select_best_pharmacist(available, shift_type, date, is_day_before_problem_day)
                    pharmacist_to_assign = chosen['name']
                    schedule_dict[date][shift_type] = pharmacist_to_assign
                    self._update_shift_counts(pharmacist_to_assign, shift_type)
                    pharmacist_hours[pharmacist_to_assign] += self.shift_types[shift_type]['hours']
                else:
                    # Rescue/SWAP step: สำหรับเวรที่ใช้ skill เฉพาะ เช่น Mixing/Care
                    # ถ้าคนมี skill ติดเวรอื่นอยู่ ให้ลองดึงคนนั้นมาลงเวรนี้
                    # แล้วหาคนอื่นที่เหมาะสมไปแทนเวรเดิมของเขา
                    rescued = self._try_rescue_assign_with_swap(
                        shuffled_pharmacists,
                        date,
                        shift_type,
                        schedule_dict,
                        pharmacist_hours,
                        pharmacist_consecutive_days,
                        is_day_before_problem_day
                    )

                    if not rescued:
                        schedule_dict[date][shift_type] = 'UNFILLED'
                        # บันทึกข้อมูลเวรที่ว่างไว้ แต่ปล่อยให้ลูปเดินหน้าต่อไปจนจบเดือน
                        if date in self.problem_days:
                            unfilled_info['problem_days'].append((date, shift_type))
                        else:
                            unfilled_info['other_days'].append((date, shift_type))

        self._fix_post_night_violations(schedule_dict, pharmacist_hours, pharmacist_consecutive_days, shuffled_pharmacists)
        # เมื่อจัดครบทุกวันแล้วค่อยสร้าง DataFrame
        final_schedule = pd.DataFrame.from_dict(schedule_dict, orient='index')
        final_schedule = final_schedule.reindex(columns=list(self.shift_types.keys()), fill_value='NO SHIFT')
        final_schedule.fillna('NO SHIFT', inplace=True)
        return final_schedule, unfilled_info

    def _update_shift_counts(self, pharmacist, shift_type):
        if self.is_night_shift(shift_type):
            self.pharmacists[pharmacist]['night_shift_count'] += 1
        if shift_type.startswith('C8'):
            self.pharmacists[pharmacist]['mixing_shift_count'] += 1
        if shift_type.startswith('Care'):
            self.pharmacists[pharmacist]['care_shift_count'] += 1
        category = self._get_shift_category(shift_type)
        if category and category in self.pharmacists[pharmacist]['category_counts']:
            self.pharmacists[pharmacist]['category_counts'][category] += 1

    def _fix_post_night_violations(self, schedule_dict, pharmacist_hours, pharmacist_consecutive_days, pharmacist_list):
        """Post-processing: swap out any shift assigned the day after a night shift."""
        dates = sorted(schedule_dict.keys())
        fixed = unfixed = 0
        for i in range(1, len(dates)):
            date, prev_date = dates[i], dates[i - 1]
            night_workers = {
                p for s, p in schedule_dict[prev_date].items()
                if p in self.pharmacists and self.is_night_shift(s)
            }
            if not night_workers:
                continue
            for shift_type in list(schedule_dict[date].keys()):
                victim = schedule_dict[date][shift_type]
                if victim not in night_workers:
                    continue
                temp_schedule = {d: dict(s) for d, s in schedule_dict.items()}
                temp_schedule[date][shift_type] = 'NO SHIFT'
                temp_hours = pharmacist_hours.copy()
                temp_hours[victim] = max(0, temp_hours.get(victim, 0) - self.shift_types[shift_type]['hours'])
                pool = [p for p in pharmacist_list if p != victim]
                candidates = self._get_available_pharmacists_optimized(
                    pool, date, shift_type, temp_schedule, temp_hours, pharmacist_consecutive_days
                )
                if candidates:
                    rep = self._select_best_pharmacist(candidates, shift_type, date, False)
                    rep_name = rep['name']
                    schedule_dict[date][shift_type] = rep_name
                    self._revert_shift_counts(victim, shift_type)
                    pharmacist_hours[victim] = max(0, pharmacist_hours.get(victim, 0) - self.shift_types[shift_type]['hours'])
                    self._update_shift_counts(rep_name, shift_type)
                    pharmacist_hours[rep_name] = pharmacist_hours.get(rep_name, 0) + self.shift_types[shift_type]['hours']
                    print(f"POST-NIGHT FIX {date.strftime('%Y-%m-%d')}: {victim} -> {shift_type} replaced by {rep_name}")
                    fixed += 1
                else:
                    schedule_dict[date][shift_type] = 'UNFILLED'
                    self._revert_shift_counts(victim, shift_type)
                    pharmacist_hours[victim] = max(0, pharmacist_hours.get(victim, 0) - self.shift_types[shift_type]['hours'])
                    print(f"POST-NIGHT UNFILLED {date.strftime('%Y-%m-%d')}: no replacement for {shift_type} (removed {victim})")
                    unfixed += 1
        if fixed + unfixed > 0:
            print(f"Post-night fix: {fixed} swapped, {unfixed} UNFILLED")

    def _revert_shift_counts(self, pharmacist, shift_type):
        """Undo shift counters when a pharmacist is moved out from an already assigned shift."""
        if pharmacist not in self.pharmacists or shift_type not in self.shift_types:
            return

        if self.is_night_shift(shift_type):
            self.pharmacists[pharmacist]['night_shift_count'] = max(
                0, self.pharmacists[pharmacist].get('night_shift_count', 0) - 1
            )
        if shift_type.startswith('C8'):
            self.pharmacists[pharmacist]['mixing_shift_count'] = max(
                0, self.pharmacists[pharmacist].get('mixing_shift_count', 0) - 1
            )
        if shift_type.startswith('Care'):
            self.pharmacists[pharmacist]['care_shift_count'] = max(
                0, self.pharmacists[pharmacist].get('care_shift_count', 0) - 1
            )

        category = self._get_shift_category(shift_type)
        if category and category in self.pharmacists[pharmacist].get('category_counts', {}):
            self.pharmacists[pharmacist]['category_counts'][category] = max(
                0, self.pharmacists[pharmacist]['category_counts'].get(category, 0) - 1
            )

    def _is_preassigned_shift(self, pharmacist, date, shift_type):
        """Return True if this exact assignment came from PreAssignments and should not be moved by rescue swap."""
        date_str = pd.to_datetime(date).strftime('%Y-%m-%d')
        return (
            pharmacist in self.pre_assignments
            and date_str in self.pre_assignments[pharmacist]
            and shift_type in self.pre_assignments[pharmacist][date_str]
        )

    def _is_skill_scarce_shift(self, shift_type):
        """Shifts with specific required skills should get rescue/swap priority."""
        required_skills = [
            str(s).strip().lower()
            for s in self.shift_types.get(shift_type, {}).get('required_skills', [])
            if str(s).strip()
        ]
        return bool(required_skills) or shift_type.startswith('C8') or shift_type.startswith('Care')

    def _can_replace_shift_after_temp_move(
        self,
        candidate,
        date,
        old_shift,
        schedule_dict,
        temp_hours,
        consecutive_days_dict
    ):
        """Check if candidate can cover old_shift after donor has been removed from it."""
        available = self._get_available_pharmacists_optimized(
            [candidate],
            date,
            old_shift,
            schedule_dict,
            temp_hours,
            consecutive_days_dict
        )
        return available[0] if available else None

    def _try_rescue_assign_with_swap(
        self,
        pharmacists,
        date,
        target_shift,
        schedule_dict,
        pharmacist_hours,
        pharmacist_consecutive_days,
        is_day_before_problem_day=False
    ):
        """
        Rescue logic for skill-scarce shifts.

        Use case:
        - target_shift เช่น C8/Mixing หรือ Care ต้องใช้ skill เฉพาะ
        - ไม่มีคนว่างเพราะคนที่มี skill ติดเวรอื่นในวันเดียวกัน
        - ระบบจะลองย้ายคนมี skill จากเวรเดิมมาลง target_shift
        - แล้วหาคนอื่นที่ทำเวรเดิมแทนได้

        This is intentionally conservative:
        - ไม่ย้ายเวรที่มาจาก PreAssignments
        - ไม่ย้ายจากเวร night
        - ไม่ย้ายจากเวร skill-scarce อีกเวรหนึ่งถ้ามีทางเลือกอื่น
        - ใช้ availability checker เดิมเพื่อลดการกระทบ constraint อื่น
        """
        if not self._is_skill_scarce_shift(target_shift):
            return False

        if not self.is_shift_available_on_date(target_shift, date):
            return False

        required_skills = [
            str(s).strip().lower()
            for s in self.shift_types[target_shift].get('required_skills', [])
            if str(s).strip()
        ]

        rescue_options = []

        # 1) หา donor: คนที่มี skill target แต่ติดเวรอื่นอยู่ในวันเดียวกัน
        for old_shift, donor in list(schedule_dict[date].items()):
            if donor not in self.pharmacists:
                continue
            if old_shift == target_shift:
                continue
            if schedule_dict[date].get(target_shift) not in ['NO SHIFT', 'UNASSIGNED', 'UNFILLED']:
                continue
            if self._is_preassigned_shift(donor, date, old_shift):
                continue
            if self.is_night_shift(old_shift):
                continue

            donor_skills = [s.strip().lower() for s in self.pharmacists[donor].get('skills', [])]
            if required_skills and not all(skill in donor_skills for skill in required_skills):
                continue

            # 2) ลอง remove donor จาก old_shift ชั่วคราว เพื่อเช็กว่า donor ลง target_shift ได้จริงหรือไม่
            original_old_assignee = schedule_dict[date][old_shift]
            original_target_assignee = schedule_dict[date][target_shift]
            schedule_dict[date][old_shift] = 'NO SHIFT'
            schedule_dict[date][target_shift] = 'NO SHIFT'

            temp_hours = pharmacist_hours.copy()
            temp_hours[donor] = max(0, temp_hours.get(donor, 0) - self.shift_types[old_shift]['hours'])

            donor_available = self._get_available_pharmacists_optimized(
                [donor],
                date,
                target_shift,
                schedule_dict,
                temp_hours,
                pharmacist_consecutive_days
            )

            if donor_available:
                # 3) หา replacement สำหรับ old_shift
                replacement_pool = [p for p in pharmacists if p != donor]
                replacement_candidates = self._get_available_pharmacists_optimized(
                    replacement_pool,
                    date,
                    old_shift,
                    schedule_dict,
                    temp_hours,
                    pharmacist_consecutive_days
                )

                if replacement_candidates:
                    replacement = self._select_best_pharmacist(
                        replacement_candidates,
                        old_shift,
                        date,
                        is_day_before_problem_day
                    )

                    old_shift_required_skills = [
                        str(s).strip().lower()
                        for s in self.shift_types[old_shift].get('required_skills', [])
                        if str(s).strip()
                    ]

                    # ยิ่ง old_shift ใช้ skill น้อย ยิ่งเหมาะกับการถูกดึง donor ออก
                    old_shift_scarcity_penalty = 1000 if self._is_skill_scarce_shift(old_shift) else 0
                    old_shift_skill_penalty = len(old_shift_required_skills) * 200
                    # Bonus if donor is missing this shift type (diversity boost)
                    donor_missing_bonus = -5000 if self.pharmacists[donor]['shift_counts'].get(target_shift, 0) == 0 else 0
                    donor_target_score = self._calculate_suitability_score(donor_available[0])
                    replacement_score = self._calculate_suitability_score(replacement)
                    total_score = (
                        old_shift_scarcity_penalty
                        + old_shift_skill_penalty
                        + donor_target_score
                        + replacement_score
                        + donor_missing_bonus
                    )

                    rescue_options.append({
                        'score': total_score,
                        'donor': donor,
                        'donor_from_shift': old_shift,
                        'target_shift': target_shift,
                        'replacement': replacement['name'],
                        'replacement_to_shift': old_shift,
                    })

            # restore temporary schedule
            schedule_dict[date][old_shift] = original_old_assignee
            schedule_dict[date][target_shift] = original_target_assignee

        if not rescue_options:
            return False

        # เลือก swap ที่กระทบ constraint น้อยที่สุด
        best = min(rescue_options, key=lambda x: x['score'])
        donor = best['donor']
        old_shift = best['donor_from_shift']
        replacement = best['replacement']

        # Apply swap จริง
        schedule_dict[date][old_shift] = replacement
        schedule_dict[date][target_shift] = donor

        self._revert_shift_counts(donor, old_shift)
        pharmacist_hours[donor] = max(0, pharmacist_hours.get(donor, 0) - self.shift_types[old_shift]['hours'])

        self._update_shift_counts(donor, target_shift)
        pharmacist_hours[donor] = pharmacist_hours.get(donor, 0) + self.shift_types[target_shift]['hours']

        self._update_shift_counts(replacement, old_shift)
        pharmacist_hours[replacement] = pharmacist_hours.get(replacement, 0) + self.shift_types[old_shift]['hours']

        print(
            f"RESCUE SWAP {date.strftime('%Y-%m-%d')}: "
            f"{donor} moved {old_shift} -> {target_shift}; "
            f"{replacement} assigned to {old_shift}"
        )
        return True

    def _get_available_pharmacists_optimized(self, pharmacists, date, shift_type, schedule_dict, current_hours_dict, consecutive_days_dict, ignore_max_hours=False):
        available_pharmacists = []
        pharmacists_on_night_yesterday = set()
        previous_date = date - timedelta(days=1)
        if previous_date in schedule_dict:
            pharmacists_on_night_yesterday = {
                p for s, p in schedule_dict[previous_date].items()
                if p in self.pharmacists and self.is_night_shift(s)
            }

        new_start = self.shift_types[shift_type]['start_time']
        new_end = self.shift_types[shift_type]['end_time']
        new_dept = self.get_department_from_shift(shift_type)

        for pharmacist in pharmacists:
            if date.strftime('%Y-%m-%d') in self.pharmacists[pharmacist]['holidays']: continue
            if self.has_overlapping_shift_optimized(pharmacist, date, shift_type, schedule_dict): continue
            if pharmacist in pharmacists_on_night_yesterday: continue
            p_skills = [skill.strip().lower() for skill in self.pharmacists[pharmacist]['skills']]
            s_req_skills = [skill.strip().lower() for skill in self.shift_types[shift_type]['required_skills'] if skill.strip()]
            if not all(skill in p_skills for skill in s_req_skills): continue

            projected_hours = current_hours_dict[pharmacist] + self.shift_types[shift_type]['hours']
            if not ignore_max_hours and projected_hours > self.pharmacists[pharmacist].get('max_hours', 250): continue
            if self.has_restricted_sequence_optimized(pharmacist, date, shift_type, schedule_dict): continue

            # --- START: Junior Constraint Logic ---
            is_junior = 'junior' in p_skills
            if is_junior:
                junior_conflict = False

                # --- 1. Department Level Check (เช็คโควต้าพื้นฐานของแต่ละแผนกตามเดิม) ---
                current_juniors_in_dept = 0
                total_dept_shifts_at_time = 0

                for s_type, s_info in self.shift_types.items():
                    if self.get_department_from_shift(s_type) == new_dept and self.is_shift_available_on_date(s_type, date):
                        s_start = s_info['start_time']
                        s_end = s_info['end_time']
                        if self.check_time_overlap(new_start, new_end, s_start, s_end):
                            total_dept_shifts_at_time += 1

                for existing_shift, assigned_pharm in schedule_dict[date].items():
                    if assigned_pharm in self.pharmacists:
                        existing_is_junior = 'junior' in [s.strip().lower() for s in self.pharmacists[assigned_pharm]['skills']]
                        if existing_is_junior:
                            existing_dept = self.get_department_from_shift(existing_shift)
                            if new_dept == existing_dept:
                                existing_start = self.shift_types[existing_shift]['start_time']
                                existing_end = self.shift_types[existing_shift]['end_time']
                                if self.check_time_overlap(new_start, new_end, existing_start, existing_end):
                                    current_juniors_in_dept += 1

                max_juniors_allowed = 2 if total_dept_shifts_at_time >= 4 else 1

                if shift_type in ['O400F2-8/1', 'O400F2-8/2', 'O400F2-8/3']:
                    max_juniors_allowed = 2

                if shift_type in ['Care/1', 'Care/2']:
                    max_juniors_allowed = 2

                if current_juniors_in_dept + 1 > max_juniors_allowed:
                    junior_conflict = True

                # --- 2. Specific Pair Check (ห้าม Junior คู่กันในเวรที่กำหนด) ---
                if not junior_conflict:
                    # คู่ที่ 1: O400F2-6 กับ O400ER-6
                    pair1 = ('O400F2-6', 'O400ER-6')
                    if shift_type in pair1:
                        other_shift = pair1[0] if shift_type == pair1[1] else pair1[1]
                        assigned_pharm = schedule_dict[date].get(other_shift)

                        # ถ้าเวรคู่กันถูกจัดไปแล้ว ให้เช็กว่าเป็น Junior หรือไม่
                        if assigned_pharm in self.pharmacists:
                            other_is_junior = 'junior' in [s.strip().lower() for s in self.pharmacists[assigned_pharm]['skills']]
                            if other_is_junior:
                                junior_conflict = True

                    # คู่ที่ 2: I100-6 กับ I400-6
                    pair2 = ('I100-6', 'I400-6')
                    if shift_type in pair2:
                        other_shift = pair2[0] if shift_type == pair2[1] else pair2[1]
                        assigned_pharm = schedule_dict[date].get(other_shift)

                        if assigned_pharm in self.pharmacists:
                            other_is_junior = 'junior' in [s.strip().lower() for s in self.pharmacists[assigned_pharm]['skills']]
                            if other_is_junior:
                                junior_conflict = True

                if junior_conflict:
                    continue
            # --- END: Junior Constraint Logic ---

            category = self._get_shift_category(shift_type)
            if category:
                limit = self.shift_limits.get(pharmacist, {}).get(category)
                if limit is not None:
                    current_count = self.pharmacists[pharmacist]['category_counts'][category]
                    if current_count >= limit:
                        continue
            if self.is_night_shift(shift_type):
                if self.has_nearby_night_shift_optimized(pharmacist, date, schedule_dict): continue
                next_date = date + timedelta(days=1)
                if pharmacist in self.pre_assignments and next_date.strftime('%Y-%m-%d') in self.pre_assignments[pharmacist]: continue
                if next_date in schedule_dict and any(v == pharmacist for v in schedule_dict[next_date].values() if v not in ('NO SHIFT', 'UNFILLED', 'UNASSIGNED')):
                    continue
            if shift_type.startswith('C8'):
                if not self.check_mixing_expert_ratio_optimized(schedule_dict, date, shift_type, pharmacist):
                    continue
            current_streak = self.get_dynamic_consecutive_days(pharmacist, date, schedule_dict)

            # Hard Constraint: ถ้าบวกเวรนี้เข้าไปแล้วทำงานติดกันเกินเพดาน ให้ตัดชื่อทิ้งทันที
            if current_streak >= self.MAX_CONSECUTIVE_DAYS:
                continue

            original_preference = self.get_preference_score(pharmacist, shift_type)
            multiplier = self.preference_multipliers.get(pharmacist, 1.0)

            # --- START: New Pacing Metrics ---
            max_hrs = self.pharmacists[pharmacist].get('max_hours', 250)
            current_hrs = current_hours_dict[pharmacist]
            _ideal_map = self._calculate_ideal_hours(date.year, date.month)
            ideal_hrs = max(_ideal_map.get(pharmacist, max_hrs), 1)

            days_in_month = pd.Timestamp(date).days_in_month
            time_elapsed_pct = date.day / days_in_month
            hours_used_pct = current_hrs / ideal_hrs
            # --- END: New Pacing Metrics ---

            # --- START: New Soul Mate & Weekend Prep ---
            is_weekend = date.weekday() >= 5
            weekend_days_worked = 0
            if is_weekend:
                weekend_days_worked = self.get_weekend_days_worked(pharmacist, schedule_dict)

            soulmate = self.soul_mates.get(pharmacist)
            soulmate_working_today = False
            mate_on_holiday = False

            if soulmate:
                # เช็คว่าคู่หูถูกจัดให้ลงเวรในวันนี้ไปหรือยัง
                if soulmate in schedule_dict[date].values():
                    soulmate_working_today = True
                # เช็คว่าวันนี้คู่หูลาหยุด (Holiday) หรือไม่
                if soulmate in self.pharmacists and date.strftime('%Y-%m-%d') in self.pharmacists[soulmate]['holidays']:
                    mate_on_holiday = True
            # --- END: New Soul Mate & Weekend Prep ---

            pharmacist_data = {
                'name': pharmacist,
                'preference_score': original_preference * multiplier,
                'consecutive_days': current_streak,
                'night_count': self.pharmacists[pharmacist]['night_shift_count'],
                'mixing_count': self.pharmacists[pharmacist]['mixing_shift_count'],
                'care_count': self.pharmacists[pharmacist].get('care_shift_count', 0),
                'current_hours': current_hrs,
                'max_hours': max_hrs,
                'time_elapsed_pct': time_elapsed_pct,
                'hours_used_pct': hours_used_pct,

                # นำตัวแปรใหม่ส่งเข้าไปให้ฟังก์ชันคิดคะแนน
                'is_weekend': is_weekend,
                'weekend_days_worked': weekend_days_worked,
                'has_soulmate': bool(soulmate),
                'soulmate_working_today': soulmate_working_today,
                'mate_on_holiday': mate_on_holiday,
                'needs_min_shift': self._needs_min_shift(pharmacist, shift_type, schedule_dict),
                'no_preference': self.pharmacists[pharmacist].get('no_preference', False),
                'department_count': self.get_dept_shift_count(pharmacist, new_dept, schedule_dict) if new_dept else 0,
                'total_shift_count': self.get_total_shift_count_in_schedule_dict(pharmacist, schedule_dict),
                'has_worked_this_department': (new_dept in self.get_unique_departments_worked(pharmacist, schedule_dict)) if new_dept else True,
                'average_monthly_shift_target': self.get_average_monthly_shift_target(date.year, date.month),
                'month_segment_shift_count': self.get_month_segment_shift_count(pharmacist, date, schedule_dict),
                'total_weekend_days': self.get_total_weekend_days_in_schedule(schedule_dict),
                'weekend_days_worked_before': self.get_weekend_days_worked_until(pharmacist, schedule_dict),
                'shift_type_count': self.pharmacists[pharmacist]['shift_counts'].get(shift_type, 0),
            }
            available_pharmacists.append(pharmacist_data)
        return available_pharmacists
    def validate_min_shift_requirements(self, schedule):
        """
        ตรวจสอบหลัง generate ว่า pharmacist คนไหนยังไม่ถึง MinCount
        คืนค่า list of (pharmacist, department, required, actual)
        """
        violations = []
        schedule_dict = {date: schedule.loc[date].to_dict() for date in schedule.index}

        for pharmacist, dept_reqs in self.min_shift_requirements.items():
            for department, min_count in dept_reqs.items():
                actual = self.get_dept_shift_count(pharmacist, department, schedule_dict)
                if actual < min_count:
                    violations.append({
                        'pharmacist': pharmacist,
                        'department': department,
                        'required': min_count,
                        'actual': actual,
                        'shortfall': min_count - actual
                    })

        if violations:
            print("\n⚠️  MIN SHIFT REQUIREMENT VIOLATIONS:")
            print(f"{'Pharmacist':<30} {'Dept':<12} {'Required':>8} {'Actual':>8} {'Short':>8}")
            print("-" * 70)
            for v in violations:
                print(f"{v['pharmacist']:<30} {v['department']:<12} "
                      f"{v['required']:>8} {v['actual']:>8} {v['shortfall']:>8}")
        else:
            print("\n✅ All minimum shift requirements are satisfied.")

        return violations
    def _calculate_suitability_score(self, pharmacist_data):
        """
        Two groups (lower score = better candidate):
          With preference   : hours(1/3) + preference(1/3) + weekend_off(1/3)
          Without preference: hours(1/3) + shift_code_distribution(1/3) + weekend_off(1/3)
        hours_used_pct = current_hrs / ideal_hrs  (ideal computed from monthly total + caps)
        """
        hours_score = pharmacist_data['hours_used_pct']

        total_weekend_days = max(pharmacist_data.get('total_weekend_days', 1), 1)
        weekend_score = pharmacist_data.get('weekend_days_worked_before', 0) / total_weekend_days

        if pharmacist_data.get('no_preference', False):
            shift_type_count = pharmacist_data.get('shift_type_count', 0)
            total_shifts     = max(pharmacist_data.get('total_shift_count', 0), 1)
            mid_score = shift_type_count / total_shifts
        else:
            raw_pref  = pharmacist_data['preference_score']
            mid_score = (raw_pref - 1) / 8.0

        score = (hours_score + mid_score + weekend_score) / 3.0

        consecutive_guard = 0.05 * (pharmacist_data['consecutive_days'] ** 2) / 9.0
        min_shift_bonus   = -0.2 if pharmacist_data.get('needs_min_shift', False) else 0.0

        pacing_guard = 0.0
        if pharmacist_data['hours_used_pct'] > pharmacist_data['time_elapsed_pct'] + 0.12:
            pacing_guard = 0.15 * (pharmacist_data['hours_used_pct'] - pharmacist_data['time_elapsed_pct'])

        return score + consecutive_guard + min_shift_bonus + pacing_guard

    def _select_best_pharmacist(self, available_pharmacists, shift_type, date, is_day_before_problem_day):
        if self.is_night_shift(shift_type) and is_day_before_problem_day:
            problem_day = date + timedelta(days=1)
            problem_day_str = problem_day.strftime('%Y-%m-%d')

            candidates_off_tomorrow = []
            for p_data in available_pharmacists:
                p_name = p_data['name']
                if problem_day_str in self.pharmacists[p_name]['holidays']:
                    candidates_off_tomorrow.append(p_data)

            if candidates_off_tomorrow:
                print(f"INFO: Prioritizing night shift on {date.strftime('%Y-%m-%d')} for pharmacists off on problem day {problem_day_str}.")
                return min(candidates_off_tomorrow, key=lambda x: (x['night_count'], self._calculate_suitability_score(x)))

        if self.is_night_shift(shift_type):
            return min(available_pharmacists, key=lambda x: (x['night_count'], self._calculate_suitability_score(x)))
        elif shift_type.startswith('C8'):
            return min(available_pharmacists, key=lambda x: (x['mixing_count'], self._calculate_suitability_score(x)))
        elif shift_type.startswith('Care'):                                                        # ← เพิ่ม
            return min(available_pharmacists, key=lambda x: (x['care_count'], self._calculate_suitability_score(x)))  # ← เพิ่ม
        else:
            return min(available_pharmacists, key=lambda x: self._calculate_suitability_score(x))

    def calculate_preference_penalty(self, pharmacist, schedule):
        penalty = 0
        for date in schedule.index:
            for shift_type, assigned_pharm in schedule.loc[date].items():
                if assigned_pharm == pharmacist:
                    penalty += self.get_preference_score(pharmacist, shift_type)
        return penalty

    def get_dynamic_consecutive_days(self, pharmacist, date, schedule_dict):
        """นับจำนวนวันทำงานต่อเนื่อง โดยกวาดดูทั้งอดีตและอนาคตจาก Schedule ปัจจุบัน"""
        streak = 0

        # เช็คย้อนหลัง (Backward)
        curr_date = date - timedelta(days=1)
        while curr_date in schedule_dict:
            worked_backward = any(p == pharmacist for p in schedule_dict[curr_date].values() if p not in ['NO SHIFT', 'UNFILLED', 'UNASSIGNED'])
            if worked_backward:
                streak += 1
                curr_date -= timedelta(days=1)
            else:
                break

        # เช็คเดินหน้า (Forward - จำเป็นเพราะเราจัด Problem Days ก่อน)
        curr_date = date + timedelta(days=1)
        while curr_date in schedule_dict:
            worked_forward = any(p == pharmacist for p in schedule_dict[curr_date].values() if p not in ['NO SHIFT', 'UNFILLED', 'UNASSIGNED'])
            if worked_forward:
                streak += 1
                curr_date += timedelta(days=1)
            else:
                break

        return streak

    def get_weekend_days_worked(self, pharmacist, schedule_dict):
        """นับจำนวนวันเสาร์-อาทิตย์ที่ทำงานไปแล้วในเดือนนี้"""
        weekend_days = 0
        for d, shifts in schedule_dict.items():
            if d.weekday() >= 5 and pharmacist in shifts.values():
                weekend_days += 1
        return weekend_days

    def _check_scheduling_requirements(self, schedule, year, month):
        """
        Returns (met: bool, issues: list[str]).
        Req 1: max-min hours among employees without a custom max_hours <= MAX_HOUR_DIFF (16)
        Req 2: max-min Preference Score (%) <= 20 (only when preferences are in use)
        """
        DEFAULT_MAX = 250
        issues = []
        uncapped = [p for p in self.pharmacists if self.pharmacists[p].get('max_hours', DEFAULT_MAX) >= DEFAULT_MAX]
        if len(uncapped) >= 2:
            hrs = {p: self.calculate_total_hours(p, schedule) for p in uncapped}
            h_max, h_min = max(hrs.values()), min(hrs.values())
            if h_max - h_min > self.MAX_HOUR_DIFF:
                hi = max(hrs, key=hrs.get)
                lo = min(hrs, key=hrs.get)
                issues.append(
                    f'Hour spread {h_max - h_min:.1f}h > {self.MAX_HOUR_DIFF}h '
                    f'(max={h_max:.1f} [{hi}], min={h_min:.1f} [{lo}])'
                )
        has_prefs = any(not self.pharmacists[p].get('no_preference', False) for p in self.pharmacists)
        if has_prefs:
            pscores = self.calculate_pharmacist_preference_scores(schedule)
            p_max, p_min = max(pscores.values()), min(pscores.values())
            if p_max - p_min > 20.0:
                hi = max(pscores, key=pscores.get)
                lo = min(pscores, key=pscores.get)
                issues.append(
                    f'Preference score spread {p_max - p_min:.1f}% > 20% '
                    f'(max={p_max:.1f}% [{hi}], min={p_min:.1f}% [{lo}])'
                )
        return len(issues) == 0, issues

    def is_schedule_better(self, current_metrics, best_metrics):
        current_unfilled = current_metrics.get('unfilled_problem_shifts', float('inf'))
        best_unfilled = best_metrics.get('unfilled_problem_shifts', float('inf'))
        if current_unfilled < best_unfilled: return True
        if current_unfilled > best_unfilled: return False
        weights = {
            'preference_score': 1.0,
            'preference_variance': 50.0,
            'hour_imbalance_penalty': 25.0,
            'night_variance': 800.0,
            'weekend_off_variance': 1000.0,
            'weekend_min_off_shortfall': 5000.0,
            'month_segment_variance': 2500.0,
        }
        current_score = sum(weights[k] * current_metrics.get(k, 0) for k in weights)
        best_score = sum(weights[k] * best_metrics.get(k, 0) for k in weights)
        return current_score < best_score

    def optimize_schedule(self, year, month, iterations=10, true_random_override=False, enable_run_log=False):
        self.run_logs = []
        self.run_config = {
            "Year": year,
            "Month": month,
            "Iterations": iterations,
            "True Random Override": true_random_override,
            "Enable Run Log": enable_run_log,
            "Staff Type": self.staff_type,
            "Run At": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        self._log_schedule_event(
            "RUN_START",
            "Start schedule generation",
            Year=year,
            Month=month,
            Iterations=iterations,
            TrueRandomOverride=true_random_override,
            EnableRunLog=enable_run_log
        )

        if true_random_override:
            print("\n⚠️ TRUE RANDOM OVERRIDE MODE ENABLED")
            print("ระบบจะสุ่มจัดเวรแบบ constrained + fairness-aware random")
            print("ยังคงบังคับ: เวรเปิดจริง, ไม่ overwrite PreAssignments, ไม่จัดหลายเวรในวันเดียว, ไม่จัดคนลา, ต้องมี skill ตรง")
            print("ยังคงคุม Fairness / Hour Balance: สุ่มจากกลุ่มคนที่ชั่วโมงน้อยหรือใกล้เคียงคนชั่วโมงน้อยก่อน")
            print("iterations ยังใช้ได้: ระบบจะสุ่มหลายรอบและเลือกตารางที่ hour balance ดีกว่า")

            best_schedule = None
            best_unfilled_info = {}
            best_metrics = {
                'unfilled_problem_shifts': float('inf'),
                'hour_imbalance_penalty': float('inf'),
                'hour_diff_for_logging': float('inf')
            }

            for i in range(iterations):
                print(f"\n--- Fair Random Iteration {i + 1}/{iterations} ---")
                current_schedule, unfilled_info = self.generate_monthly_schedule_true_random(
                    year=year,
                    month=month,
                    iteration_num=i + 1
                )

                metrics = self.calculate_schedule_metrics(current_schedule, year, month)
                metrics['unfilled_problem_shifts'] = (
                    len(unfilled_info.get('problem_days', [])) + len(unfilled_info.get('other_days', []))
                )

                print(
                    f"Fair Random Results -> "
                    f"Unfilled Shifts: {metrics['unfilled_problem_shifts']} | "
                    f"Hour SD: {metrics.get('hour_diff_for_logging', 0):.2f} | "
                    f"Hour Penalty: {metrics.get('hour_imbalance_penalty', 0):.2f}"
                )

                self._log_schedule_event(
                    "TRUE_RANDOM_ITERATION_RESULT",
                    "Completed one fairness-aware random iteration",
                    Iteration=i + 1,
                    UnfilledShifts=metrics['unfilled_problem_shifts'],
                    HourSD=round(metrics.get('hour_diff_for_logging', 0), 4),
                    HourPenalty=round(metrics.get('hour_imbalance_penalty', 0), 4)
                )

                current_key = (
                    metrics['unfilled_problem_shifts'],
                    metrics.get('hour_imbalance_penalty', float('inf')),
                    metrics.get('hour_diff_for_logging', float('inf'))
                )
                best_key = (
                    best_metrics.get('unfilled_problem_shifts', float('inf')),
                    best_metrics.get('hour_imbalance_penalty', float('inf')),
                    best_metrics.get('hour_diff_for_logging', float('inf'))
                )

                if best_schedule is None or current_key < best_key:
                    best_schedule = current_schedule.copy()
                    best_unfilled_info = unfilled_info.copy()
                    best_metrics = metrics.copy()
                    print("*** Found a fairer random schedule! ***")
                    req_met, req_issues = self._check_scheduling_requirements(best_schedule, year, month)
                    if req_met:
                        print(f"✅ All scheduling requirements met at iteration {i + 1}. Stopping early.")
                        break
                    else:
                        for _iss in req_issues:
                            print(f"  ⚠️  {_iss}")

                    self._log_schedule_event(
                        "TRUE_RANDOM_BEST_UPDATED",
                        "Found a better fairness-aware random schedule",
                        Iteration=i + 1,
                        UnfilledShifts=metrics['unfilled_problem_shifts'],
                        HourSD=round(metrics.get('hour_diff_for_logging', 0), 4),
                        HourPenalty=round(metrics.get('hour_imbalance_penalty', 0), 4)
                    )

            self._log_schedule_event(
                "RUN_COMPLETE",
                "Completed fairness-aware constrained true random override mode",
                TotalUnfilled=len(best_unfilled_info.get('problem_days', [])) + len(best_unfilled_info.get('other_days', [])),
                FinalHourSD=round(best_metrics.get('hour_diff_for_logging', 0), 4),
                FinalHourPenalty=round(best_metrics.get('hour_imbalance_penalty', 0), 4)
            )

            if best_schedule is not None:
                best_schedule, best_unfilled_info = self.post_optimization_rescue_and_balance(
                    best_schedule, best_unfilled_info, year, month
                )
            return best_schedule, best_unfilled_info

        best_schedule = None
        best_metrics = {
            'unfilled_problem_shifts': float('inf'),
            'hour_imbalance_penalty': float('inf'),
            'night_variance': float('inf'),
            'preference_score': float('inf')
        }
        best_unfilled_info = {}

        self._pre_check_staffing_levels(year, month)

        _ideal = self._calculate_ideal_hours(year, month)
        _total_avail = sum(_ideal.values())
        print(f"\nTotal available shift hours ({year}-{month:02d}): {_total_avail:.1f} hrs")
        print("  Target hours per employee:")
        for _p, _h in sorted(_ideal.items(), key=lambda x: x[1]):
            _cap = self.pharmacists[_p].get('max_hours', 250)
            _cap_str = f" (cap {_cap}h)" if _cap < 250 else ""
            print(f"    {_p}: {_h:.1f}h{_cap_str}")

        print(f"\nStarting optimization with {iterations} iterations...")

        for i in range(iterations):
            print(f"\n--- Iteration {i + 1}/{iterations} ---")
            current_schedule, unfilled_info = self.generate_monthly_schedule_shuffled(
                year,
                month,
                iteration_num=i + 1
            )

            metrics = self.calculate_schedule_metrics(current_schedule, year, month)
            metrics['unfilled_problem_shifts'] = len(unfilled_info['problem_days']) + len(unfilled_info['other_days'])

            print(
                f"Iteration Results -> "
                f"Unfilled Shifts: {metrics['unfilled_problem_shifts']} | "
                f"Hour SD: {metrics.get('hour_diff_for_logging', 0):.2f} | "
                f"Night Var: {metrics.get('night_variance', 0):.2f} | "
                f"Weekend Shortfall: {metrics.get('weekend_min_off_shortfall', 0)} | "
                f"Month Segment Var: {metrics.get('month_segment_variance', 0):.2f} | "
                f"Pref Penalty: {metrics.get('preference_score', 0):.1f}"
            )

            self._log_schedule_event(
                "ITERATION_RESULT",
                "Completed one optimization iteration",
                Iteration=i + 1,
                UnfilledShifts=metrics['unfilled_problem_shifts'],
                HourSD=round(metrics.get('hour_diff_for_logging', 0), 4),
                NightVariance=round(metrics.get('night_variance', 0), 4),
                WeekendShortfall=metrics.get('weekend_min_off_shortfall', 0),
                MonthSegmentVariance=round(metrics.get('month_segment_variance', 0), 4),
                PreferencePenalty=round(metrics.get('preference_score', 0), 4)
            )

            if best_schedule is None or self.is_schedule_better(metrics, best_metrics):
                best_schedule = current_schedule.copy()
                best_metrics = metrics.copy()
                best_unfilled_info = unfilled_info.copy()
                print("*** Found a more balanced schedule! ***")
                req_met, req_issues = self._check_scheduling_requirements(best_schedule, year, month)
                if req_met:
                    print(f"✅ All scheduling requirements met at iteration {i + 1}. Stopping early.")
                    break
                else:
                    for _iss in req_issues:
                        print(f"  ⚠️  {_iss}")

                self._log_schedule_event(
                    "BEST_UPDATED",
                    "Found a better schedule",
                    Iteration=i + 1,
                    UnfilledShifts=metrics['unfilled_problem_shifts'],
                    HourSD=round(metrics.get('hour_diff_for_logging', 0), 4),
                    NightVariance=round(metrics.get('night_variance', 0), 4),
                    WeekendShortfall=metrics.get('weekend_min_off_shortfall', 0),
                    MonthSegmentVariance=round(metrics.get('month_segment_variance', 0), 4),
                    PreferencePenalty=round(metrics.get('preference_score', 0), 4)
                )

        if best_schedule is not None:
            final_req_met, final_req_issues = self._check_scheduling_requirements(best_schedule, year, month)
            if not final_req_met:
                print("⚠️  Requirements NOT fully met after all iterations:")
                for _iss in final_req_issues:
                    print(f"  - {_iss}")
            else:
                print("✅ All scheduling requirements satisfied.")
        if best_schedule is not None:
            print("\nOptimization complete!\nFinal metrics for the best schedule found:")
            print(
                f"Unfilled Shifts: {best_metrics.get('unfilled_problem_shifts', 0)} | "
                f"Hour SD: {best_metrics.get('hour_diff_for_logging', 0):.2f} | "
                f"Night Var: {best_metrics.get('night_variance', 0):.2f} | "
                f"Weekend Shortfall: {best_metrics.get('weekend_min_off_shortfall', 0)} | "
                f"Month Segment Var: {best_metrics.get('month_segment_variance', 0):.2f} | "
                f"Pref Penalty: {best_metrics.get('preference_score', 0):.1f}"
            )
            self.validate_min_shift_requirements(best_schedule)

            self._log_schedule_event(
                "RUN_COMPLETE",
                "Completed optimization mode",
                FinalUnfilledShifts=best_metrics.get('unfilled_problem_shifts', 0),
                FinalHourSD=round(best_metrics.get('hour_diff_for_logging', 0), 4),
                FinalNightVariance=round(best_metrics.get('night_variance', 0), 4),
                FinalWeekendShortfall=best_metrics.get('weekend_min_off_shortfall', 0),
                FinalMonthSegmentVariance=round(best_metrics.get('month_segment_variance', 0), 4),
                FinalPreferencePenalty=round(best_metrics.get('preference_score', 0), 4)
            )
        else:
            print("\nOptimization failed to find any valid schedule.")
            self._log_schedule_event(
                "RUN_FAILED",
                "Optimization failed to find valid schedule"
            )

        if best_schedule is not None:
            best_schedule, best_unfilled_info = self.post_optimization_rescue_and_balance(
                best_schedule, best_unfilled_info, year, month
            )
        return best_schedule, best_unfilled_info



    def post_optimization_rescue_and_balance(self, schedule, unfilled_info, year, month):
        """
        Four-phase post-optimization pass applied after the best schedule is found.

        Phase 1:  Fill remaining UNFILLED shifts via direct assignment first,
                  then rescue-swap. Problem-day shifts are processed first.
        Phase 1b: Fallback for still-UNFILLED: relax max_hours once per employee.
        Phase 2:  Preference staff preference-score improvement.
        Phase 3:  No-preference staff shift-code distribution.
        Phase 4:  Rebalance hours toward per-pharmacist ideal targets.
                  person covers every shift code they have skills for (>=1),
                  then equalise counts more strictly (highest priority).
        """
        # ── Build working structures from the final schedule ─────────────
        schedule_dict = {date: schedule.loc[date].to_dict() for date in schedule.index}

        self._reset_runtime_shift_counters()
        for date in schedule.index:
            for shift_type, assigned in schedule.loc[date].items():
                if assigned in self.pharmacists:
                    self._update_shift_counts(assigned, shift_type)

        def _recalc_hours():
            return {
                p: sum(
                    self.shift_types[st]['hours']
                    for d, shifts in schedule_dict.items()
                    for st, v in shifts.items()
                    if v == p and st in self.shift_types
                )
                for p in self.pharmacists
            }

        pharmacist_hours = _recalc_hours()
        pharmacist_list  = list(self.pharmacists.keys())

        def _consec(date):
            return {p: self.get_dynamic_consecutive_days(p, date, schedule_dict)
                    for p in self.pharmacists}

        # ── Phase 1: Fill UNFILLED shifts ────────────────────────────────
        unfilled_shifts = [
            (d, st)
            for d in sorted(schedule_dict)
            for st, v in schedule_dict[d].items()
            if v == 'UNFILLED'
        ]
        unfilled_shifts.sort(key=lambda x: (x[0] not in self.problem_days, x[0]))

        total_unfilled = len(unfilled_shifts)
        print("\n[POST-OPT] Phase 1: Attempting to fill %d UNFILLED shift(s)..." % total_unfilled)
        filled        = 0
        still_unfilled = []

        for date, shift_type in unfilled_shifts:
            schedule_dict[date][shift_type] = 'NO SHIFT'
            consec = _consec(date)

            candidates = self._get_available_pharmacists_optimized(
                pharmacist_list, date, shift_type, schedule_dict, pharmacist_hours, consec
            )
            if candidates:
                p = self._select_best_pharmacist(candidates, shift_type, date, False)['name']
                schedule_dict[date][shift_type] = p
                pharmacist_hours[p] += self.shift_types[shift_type]['hours']
                self._update_shift_counts(p, shift_type)
                filled += 1
                print("  [DIRECT] %s %s -> %s" % (date.strftime('%Y-%m-%d'), shift_type, p))
                continue

            rescued = self._try_rescue_assign_with_swap(
                pharmacist_list, date, shift_type,
                schedule_dict, pharmacist_hours, consec, False
            )
            if rescued:
                filled += 1
            else:
                schedule_dict[date][shift_type] = 'UNFILLED'
                still_unfilled.append((date, shift_type))

        print("  Filled %d/%d. %d remain unfilled." % (filled, total_unfilled, len(still_unfilled)))

        # ── Phase 1b: max-hours override fallback (once per employee) ────
        if still_unfilled:
            max_hours_override_used = set()
            new_still_unfilled = []
            print("\n[POST-OPT] Phase 1b: Max-hours override for %d remaining UNFILLED..." % len(still_unfilled))
            override_filled = 0

            for date, shift_type in still_unfilled:
                schedule_dict[date][shift_type] = 'NO SHIFT'
                consec = _consec(date)
                shift_hrs = self.shift_types[shift_type]['hours']
                override_assigned = False

                for p in pharmacist_list:
                    if p in max_hours_override_used:
                        continue
                    # Zero this person's hours so max_hours check passes
                    temp_hours = dict(pharmacist_hours)
                    temp_hours[p] = 0
                    cands = self._get_available_pharmacists_optimized(
                        [p], date, shift_type, schedule_dict, temp_hours, consec
                    )
                    if cands:
                        schedule_dict[date][shift_type] = p
                        pharmacist_hours[p] += shift_hrs
                        self._update_shift_counts(p, shift_type)
                        max_hours_override_used.add(p)
                        override_filled += 1
                        filled += 1
                        override_assigned = True
                        print("  [OVERRIDE] %s %s -> %s (max_hours bypassed)" % (
                            date.strftime('%Y-%m-%d'), shift_type, p))
                        break

                if not override_assigned:
                    schedule_dict[date][shift_type] = 'UNFILLED'
                    new_still_unfilled.append((date, shift_type))

            still_unfilled = new_still_unfilled
            print("  Override filled %d more. %d still unfilled." % (override_filled, len(still_unfilled)))

        # ── Phase 2: Preference staff preference-score improvement ────────
        pref_list = [
            p for p in pharmacist_list
            if not self.pharmacists[p].get('no_preference', False)
            and 'junior' not in [s.strip().lower() for s in self.pharmacists[p]['skills']]
        ]
        print("\n[POST-OPT] Phase 2: Preference score improvement (%d staff)..." % len(pref_list))

        if pref_list:
            phase3_swaps = 0

            for _iter in range(30):
                best_pref = None  # (gain, date, st1, st2, p1, p2)

                for date in sorted(schedule_dict):
                    shifts_today = [
                        (st, schedule_dict[date][st])
                        for st in schedule_dict[date]
                        if schedule_dict[date][st] in pref_list
                    ]
                    for i in range(len(shifts_today)):
                        st1, p1 = shifts_today[i]
                        if self._is_preassigned_shift(p1, date, st1):
                            continue
                        sc1_p1 = self.get_preference_score(p1, st1)

                        for j in range(len(shifts_today)):
                            if i == j:
                                continue
                            st2, p2 = shifts_today[j]
                            if self._is_preassigned_shift(p2, date, st2):
                                continue

                            sc2_p1 = self.get_preference_score(p1, st2)
                            sc1_p2 = self.get_preference_score(p2, st1)
                            sc2_p2 = self.get_preference_score(p2, st2)

                            gain = (sc1_p1 - sc2_p1) + (sc2_p2 - sc1_p2)
                            if gain <= 0:
                                continue

                            orig1 = schedule_dict[date][st1]
                            orig2 = schedule_dict[date][st2]
                            schedule_dict[date][st1] = 'NO SHIFT'
                            schedule_dict[date][st2] = 'NO SHIFT'
                            hrs_now = _recalc_hours()
                            consec = _consec(date)

                            c1 = self._get_available_pharmacists_optimized(
                                [p1], date, st2, schedule_dict, hrs_now, consec
                            )
                            c2 = self._get_available_pharmacists_optimized(
                                [p2], date, st1, schedule_dict, hrs_now, consec
                            )
                            schedule_dict[date][st1] = orig1
                            schedule_dict[date][st2] = orig2

                            if c1 and c2:
                                if best_pref is None or gain > best_pref[0]:
                                    best_pref = (gain, date, st1, st2, p1, p2)

                if best_pref is None:
                    break

                gain, date, st1, st2, p1, p2 = best_pref
                schedule_dict[date][st1] = p2
                schedule_dict[date][st2] = p1
                self._revert_shift_counts(p1, st1)
                self._update_shift_counts(p1, st2)
                self._revert_shift_counts(p2, st2)
                self._update_shift_counts(p2, st1)
                phase3_swaps += 1
                print("  [PREF] %s: %s(%s) <-> %s(%s)  gain=%d" % (
                    date.strftime('%Y-%m-%d'), p1, st1, p2, st2, gain))

            print("  Phase 2 total swaps: %d" % phase3_swaps)

        # ── Phase 3: No-preference shift-code distribution ───────────────
        no_pref_list = [p for p in pharmacist_list if p in self.no_preference_staff]
        print("\n[POST-OPT] Phase 3: No-preference shift-code distribution (%d staff)..." % len(no_pref_list))

        if no_pref_list:
            def _can_work_skill(p, st):
                p_skills = [s.strip().lower() for s in self.pharmacists[p]['skills']]
                s_req = [s.strip().lower() for s in self.shift_types[st].get('required_skills', []) if s.strip() and s.strip().lower() != 'nan']
                return all(sk in p_skills for sk in s_req)

            skilled_map = {p: [st for st in self.shift_types if _can_work_skill(p, st)] for p in no_pref_list}

            def _np_coverage():
                return {
                    p: {st: sum(1 for d in schedule_dict if schedule_dict[d].get(st) == p)
                        for st in self.shift_types}
                    for p in no_pref_list
                }

            phase4_swaps = 0

            # Sub-pass A: ensure >=1 of every skilled shift type
            for _iter in range(40):
                cov = _np_coverage()
                target_found = False

                for p in no_pref_list:
                    skilled = skilled_map[p]
                    missing = [st for st in skilled if cov[p][st] == 0]
                    if not missing:
                        continue

                    for want_st in missing:
                        for date in sorted(schedule_dict):
                            holder = schedule_dict[date].get(want_st)
                            if holder not in self.pharmacists or holder == p:
                                continue
                            if self._is_preassigned_shift(holder, date, want_st):
                                continue
                            if holder in self.no_preference_staff and cov[holder][want_st] <= 1:
                                continue

                            # Try direct assignment first
                            p_shifts_today = [s for s, h in schedule_dict[date].items() if h == p]
                            if not p_shifts_today:
                                schedule_dict[date][want_st] = 'NO SHIFT'
                                hrs_now = _recalc_hours()
                                consec = _consec(date)
                                cands = self._get_available_pharmacists_optimized(
                                    [p], date, want_st, schedule_dict, hrs_now, consec, ignore_max_hours=True
                                )
                                if cands:
                                    schedule_dict[date][want_st] = p
                                    self._revert_shift_counts(holder, want_st)
                                    self._update_shift_counts(p, want_st)
                                    phase4_swaps += 1
                                    target_found = True
                                    print("  [NP-COVER] %s %s: %s -> %s" % (
                                        date.strftime('%Y-%m-%d'), want_st, holder, p))
                                    break
                                else:
                                    schedule_dict[date][want_st] = holder
                            else:
                                # Rescue Swap logic
                                p_old_shift = p_shifts_today[0]
                                if self._is_preassigned_shift(p, date, p_old_shift) or self.is_night_shift(p_old_shift):
                                    continue
                                
                                original_holder = schedule_dict[date][want_st]
                                original_p_assignee = schedule_dict[date][p_old_shift]
                                
                                schedule_dict[date][want_st] = 'NO SHIFT'
                                schedule_dict[date][p_old_shift] = 'NO SHIFT'
                                
                                hrs_now = _recalc_hours()
                                consec = _consec(date)
                                
                                cands_p = self._get_available_pharmacists_optimized(
                                    [p], date, want_st, schedule_dict, hrs_now, consec, ignore_max_hours=True
                                )
                                if cands_p:
                                    rep_pool = [ph for ph in pharmacist_list if ph != p]
                                    cands_rep = self._get_available_pharmacists_optimized(
                                        rep_pool, date, p_old_shift, schedule_dict, hrs_now, consec, ignore_max_hours=True
                                    )
                                    if cands_rep:
                                        rep = self._select_best_pharmacist(cands_rep, p_old_shift, date, False)['name']
                                        schedule_dict[date][want_st] = p
                                        schedule_dict[date][p_old_shift] = rep
                                        
                                        self._revert_shift_counts(holder, want_st)
                                        self._revert_shift_counts(p, p_old_shift)
                                        self._update_shift_counts(p, want_st)
                                        self._update_shift_counts(rep, p_old_shift)
                                        
                                        phase4_swaps += 1
                                        target_found = True
                                        print("  [NP-RESCUE] %s: %s -> %s, %s -> %s (for %s)" % (
                                            date.strftime('%Y-%m-%d'), holder, p, p, rep, want_st))
                                        break
                                
                                # Restore
                                schedule_dict[date][want_st] = original_holder
                                schedule_dict[date][p_old_shift] = original_p_assignee

                        if target_found:
                            break
                    if target_found:
                        break

                if not target_found:
                    break

            # Sub-pass B: equalise distribution (stricter)
            for _iter in range(60):
                cov = _np_coverage()
                best_eq = None  # (gap, date, st, p_over, p_under)

                for p1 in no_pref_list:
                    for p2 in no_pref_list:
                        if p1 == p2:
                            continue
                        for st in self.shift_types:
                            if st not in skilled_map[p1] or st not in skilled_map[p2]:
                                continue
                            p1_cnt = cov[p1].get(st, 0)
                            p2_cnt = cov[p2].get(st, 0)
                            gap = p1_cnt - p2_cnt
                            if gap < 2:
                                continue
                            for date in sorted(schedule_dict):
                                if schedule_dict[date].get(st) != p1:
                                    continue
                                if self._is_preassigned_shift(p1, date, st):
                                    continue
                                schedule_dict[date][st] = 'NO SHIFT'
                                hrs_now = _recalc_hours()
                                consec = _consec(date)
                                cands = self._get_available_pharmacists_optimized(
                                    [p2], date, st, schedule_dict, hrs_now, consec, ignore_max_hours=True
                                )
                                schedule_dict[date][st] = p1
                                if cands:
                                    if best_eq is None or gap > best_eq[0]:
                                        best_eq = (gap, date, st, p1, p2)
                                    break

                if best_eq is None:
                    break

                _, date, st, p_over, p_under = best_eq
                schedule_dict[date][st] = p_under
                self._revert_shift_counts(p_over, st)
                self._update_shift_counts(p_under, st)
                phase4_swaps += 1
                print("  [NP-EQUAL] %s %s: %s -> %s (gap=%d)" % (
                    date.strftime('%Y-%m-%d'), st, p_over, p_under, best_eq[0]))

            # Sub-pass C: Internal balancing for each no-preference staff
            # Ensure each person's distribution across their skilled shift codes is balanced.
            for _iter in range(40):
                cov = _np_coverage()
                target_found = False
                
                for p in no_pref_list:
                    skilled = skilled_map[p]
                    if len(skilled) < 2:
                        continue
                        
                    # Find shift types with highest and lowest counts for this person
                    p_counts = {st: cov[p][st] for st in skilled}
                    st_high = max(p_counts, key=p_counts.get)
                    st_low = min(p_counts, key=p_counts.get)
                    
                    if p_counts[st_high] - p_counts[st_low] < 2:
                        continue
                    
                    # Try to swap one st_high for st_low with another holder
                    for date in sorted(schedule_dict):
                        if schedule_dict[date].get(st_high) != p:
                            continue
                        if self._is_preassigned_shift(p, date, st_high):
                            continue
                            
                        # Someone else must hold st_low on this date
                        holder = schedule_dict[date].get(st_low)
                        if holder not in self.pharmacists or holder == p:
                            continue
                        if self._is_preassigned_shift(holder, date, st_low):
                            continue
                        
                        # Check if swap is possible (both can work each other's shifts)
                        orig_st_high_assignee = schedule_dict[date][st_high]
                        orig_st_low_assignee = schedule_dict[date][st_low]
                        
                        schedule_dict[date][st_high] = 'NO SHIFT'
                        schedule_dict[date][st_low] = 'NO SHIFT'
                        
                        hrs_now = _recalc_hours()
                        consec = _consec(date)
                        
                        # Check if p can take st_low and holder can take st_high
                        cands_p = self._get_available_pharmacists_optimized(
                            [p], date, st_low, schedule_dict, hrs_now, consec, ignore_max_hours=True
                        )
                        cands_h = self._get_available_pharmacists_optimized(
                            [holder], date, st_high, schedule_dict, hrs_now, consec, ignore_max_hours=True
                        )
                        
                        if cands_p and cands_h:
                            schedule_dict[date][st_high] = holder
                            schedule_dict[date][st_low] = p
                            
                            self._revert_shift_counts(p, st_high)
                            self._update_shift_counts(p, st_low)
                            self._revert_shift_counts(holder, st_low)
                            self._update_shift_counts(holder, st_high)
                            
                            phase4_swaps += 1
                            target_found = True
                            print("  [NP-INTERNAL] %s: %s(%s->%s), %s(%s->%s)" % (
                                date.strftime('%Y-%m-%d'), p, st_high, st_low, holder, st_low, st_high))
                            break
                        else:
                            # Restore
                            schedule_dict[date][st_high] = orig_st_high_assignee
                            schedule_dict[date][st_low] = orig_st_low_assignee
                            
                    if target_found:
                        break
                if not target_found:
                    break

            print("  Phase 3 total swaps: %d" % phase4_swaps)

        # ── Phase 4: Hour rebalancing ─────────────────────────────────────
        ideal_map = self._calculate_ideal_hours(year, month)
        print("\n[POST-OPT] Phase 4: Rebalancing hours toward ideal targets...")

        total_swaps = 0

        for pass_num in range(30):
            hours_now  = _recalc_hours()
            deviations = {p: hours_now[p] - ideal_map.get(p, 0) for p in self.pharmacists}

            over_set  = sorted(
                [p for p in self.pharmacists if deviations[p] > 0],
                key=lambda p: -deviations[p]
            )[:5]
            under_set = sorted(
                [p for p in self.pharmacists if deviations[p] < 0],
                key=lambda p: deviations[p]
            )[:5]

            if not over_set or not under_set:
                print("  Pass %d: All pharmacists are at or near ideal hours. Done." % (pass_num + 1))
                break

            best = None

            for over_p in over_set:
                for under_p in under_set:
                    gap_before = deviations[over_p] - deviations[under_p]
                    if gap_before < 1.0:
                        continue

                    for date in sorted(schedule_dict):
                        for shift_type, assigned in schedule_dict[date].items():
                            if assigned != over_p:
                                continue
                            if self._is_preassigned_shift(over_p, date, shift_type):
                                continue

                            hrs = self.shift_types.get(shift_type, {}).get('hours', 0)
                            if hrs <= 0:
                                continue

                            new_gap    = abs((deviations[over_p] - hrs) - (deviations[under_p] + hrs))
                            improvement = gap_before - new_gap
                            if improvement <= 0:
                                continue
                            if best is not None and improvement <= best[0]:
                                continue

                            original   = schedule_dict[date][shift_type]
                            schedule_dict[date][shift_type] = 'NO SHIFT'
                            temp_hours = dict(hours_now)
                            temp_hours[over_p] = hours_now[over_p] - hrs
                            consec = _consec(date)

                            cands = self._get_available_pharmacists_optimized(
                                [under_p], date, shift_type,
                                schedule_dict, temp_hours, consec
                            )
                            schedule_dict[date][shift_type] = original

                            if cands:
                                best = (improvement, date, shift_type, over_p, under_p, hrs)

            if best is None:
                print("  Pass %d: No improving swap found. Done." % (pass_num + 1))
                break

            _, date, shift_type, over_p, under_p, hrs = best
            schedule_dict[date][shift_type] = under_p
            self._revert_shift_counts(over_p, shift_type)
            self._update_shift_counts(under_p, shift_type)

            old_over  = hours_now[over_p]
            old_under = hours_now[under_p]
            total_swaps += 1
            print(
                "  Pass %d: %s %s  %s(%gh->%gh)  =>  %s(%gh->%gh)" % (
                    pass_num + 1,
                    date.strftime('%Y-%m-%d'), shift_type,
                    over_p,  old_over,  old_over  - hrs,
                    under_p, old_under, old_under + hrs,
                )
            )

        print("  Total balance moves: %d" % total_swaps)

        # ── Phase 5: No-preference shift-code distribution ───────────────
        no_pref_list = [p for p in pharmacist_list if p in self.no_preference_staff]
        print("\n[POST-OPT] Phase 5: No-preference shift-code distribution (%d staff)..." % len(no_pref_list))

        if no_pref_list:
            def _can_work_skill(p, st):
                p_skills = [s.strip().lower() for s in self.pharmacists[p]['skills']]
                s_req = [s.strip().lower() for s in self.shift_types[st].get('required_skills', []) if s.strip() and s.strip().lower() != 'nan']
                return all(sk in p_skills for sk in s_req)

            skilled_map = {p: [st for st in self.shift_types if _can_work_skill(p, st)] for p in no_pref_list}

            def _np_coverage():
                return {
                    p: {st: sum(1 for d in schedule_dict if schedule_dict[d].get(st) == p)
                        for st in self.shift_types}
                    for p in no_pref_list
                }

            phase5_swaps = 0

            # Sub-pass A: ensure >=1 of every skilled shift type
            for _iter in range(40):
                cov = _np_coverage()
                target_found = False

                for p in no_pref_list:
                    skilled = skilled_map[p]
                    missing = [st for st in skilled if cov[p][st] == 0]
                    if not missing:
                        continue

                    for want_st in missing:
                        for date in sorted(schedule_dict):
                            holder = schedule_dict[date].get(want_st)
                            if holder not in self.pharmacists or holder == p:
                                continue
                            if self._is_preassigned_shift(holder, date, want_st):
                                continue
                            if holder in self.no_preference_staff and cov[holder][want_st] <= 1:
                                continue

                            # Try direct assignment first
                            p_shifts_today = [s for s, h in schedule_dict[date].items() if h == p]
                            if not p_shifts_today:
                                schedule_dict[date][want_st] = 'NO SHIFT'
                                hrs_now = _recalc_hours()
                                consec = _consec(date)
                                cands = self._get_available_pharmacists_optimized(
                                    [p], date, want_st, schedule_dict, hrs_now, consec, ignore_max_hours=True
                                )
                                if cands:
                                    schedule_dict[date][want_st] = p
                                    self._revert_shift_counts(holder, want_st)
                                    self._update_shift_counts(p, want_st)
                                    phase5_swaps += 1
                                    target_found = True
                                    print("  [NP-COVER] %s %s: %s -> %s" % (
                                        date.strftime('%Y-%m-%d'), want_st, holder, p))
                                    break
                                else:
                                    schedule_dict[date][want_st] = holder
                            else:
                                # Rescue Swap logic
                                p_old_shift = p_shifts_today[0]
                                if self._is_preassigned_shift(p, date, p_old_shift) or self.is_night_shift(p_old_shift):
                                    continue
                                
                                original_holder = schedule_dict[date][want_st]
                                original_p_assignee = schedule_dict[date][p_old_shift]
                                
                                schedule_dict[date][want_st] = 'NO SHIFT'
                                schedule_dict[date][p_old_shift] = 'NO SHIFT'
                                
                                hrs_now = _recalc_hours()
                                consec = _consec(date)
                                
                                cands_p = self._get_available_pharmacists_optimized(
                                    [p], date, want_st, schedule_dict, hrs_now, consec, ignore_max_hours=True
                                )
                                if cands_p:
                                    rep_pool = [ph for ph in pharmacist_list if ph != p]
                                    cands_rep = self._get_available_pharmacists_optimized(
                                        rep_pool, date, p_old_shift, schedule_dict, hrs_now, consec, ignore_max_hours=True
                                    )
                                    if cands_rep:
                                        rep = self._select_best_pharmacist(cands_rep, p_old_shift, date, False)['name']
                                        schedule_dict[date][want_st] = p
                                        schedule_dict[date][p_old_shift] = rep
                                        
                                        self._revert_shift_counts(holder, want_st)
                                        self._revert_shift_counts(p, p_old_shift)
                                        self._update_shift_counts(p, want_st)
                                        self._update_shift_counts(rep, p_old_shift)
                                        
                                        phase5_swaps += 1
                                        target_found = True
                                        print("  [NP-RESCUE] %s: %s -> %s, %s -> %s (for %s)" % (
                                            date.strftime('%Y-%m-%d'), holder, p, p, rep, want_st))
                                        break
                                
                                # Restore
                                schedule_dict[date][want_st] = original_holder
                                schedule_dict[date][p_old_shift] = original_p_assignee

                        if target_found:
                            break
                    if target_found:
                        break

                if not target_found:
                    break

            # Sub-pass B: equalise distribution (stricter)
            for _iter in range(60):
                cov = _np_coverage()
                best_eq = None  # (gap, date, st, p_over, p_under)

                for p1 in no_pref_list:
                    for p2 in no_pref_list:
                        if p1 == p2:
                            continue
                        for st in self.shift_types:
                            if st not in skilled_map[p1] or st not in skilled_map[p2]:
                                continue
                            p1_cnt = cov[p1].get(st, 0)
                            p2_cnt = cov[p2].get(st, 0)
                            gap = p1_cnt - p2_cnt
                            if gap < 2:
                                continue
                            for date in sorted(schedule_dict):
                                if schedule_dict[date].get(st) != p1:
                                    continue
                                if self._is_preassigned_shift(p1, date, st):
                                    continue
                                schedule_dict[date][st] = 'NO SHIFT'
                                hrs_now = _recalc_hours()
                                consec = _consec(date)
                                cands = self._get_available_pharmacists_optimized(
                                    [p2], date, st, schedule_dict, hrs_now, consec, ignore_max_hours=True
                                )
                                schedule_dict[date][st] = p1
                                if cands:
                                    if best_eq is None or gap > best_eq[0]:
                                        best_eq = (gap, date, st, p1, p2)
                                    break

                if best_eq is None:
                    break

                _, date, st, p_over, p_under = best_eq
                schedule_dict[date][st] = p_under
                self._revert_shift_counts(p_over, st)
                self._update_shift_counts(p_under, st)
                phase5_swaps += 1
                print("  [NP-EQUAL] %s %s: %s -> %s (gap=%d)" % (
                    date.strftime('%Y-%m-%d'), st, p_over, p_under, best_eq[0]))

            # Sub-pass C: Internal balancing for each no-preference staff
            # Ensure each person's distribution across their skilled shift codes is balanced.
            for _iter in range(40):
                cov = _np_coverage()
                target_found = False
                
                for p in no_pref_list:
                    skilled = skilled_map[p]
                    if len(skilled) < 2:
                        continue
                        
                    # Find shift types with highest and lowest counts for this person
                    p_counts = {st: cov[p][st] for st in skilled}
                    st_high = max(p_counts, key=p_counts.get)
                    st_low = min(p_counts, key=p_counts.get)
                    
                    if p_counts[st_high] - p_counts[st_low] < 2:
                        continue
                    
                    # Try to swap one st_high for st_low with another holder
                    for date in sorted(schedule_dict):
                        if schedule_dict[date].get(st_high) != p:
                            continue
                        if self._is_preassigned_shift(p, date, st_high):
                            continue
                            
                        # Someone else must hold st_low on this date
                        holder = schedule_dict[date].get(st_low)
                        if holder not in self.pharmacists or holder == p:
                            continue
                        if self._is_preassigned_shift(holder, date, st_low):
                            continue
                        
                        # Check if swap is possible (both can work each other's shifts)
                        orig_st_high_assignee = schedule_dict[date][st_high]
                        orig_st_low_assignee = schedule_dict[date][st_low]
                        
                        schedule_dict[date][st_high] = 'NO SHIFT'
                        schedule_dict[date][st_low] = 'NO SHIFT'
                        
                        hrs_now = _recalc_hours()
                        consec = _consec(date)
                        
                        # Check if p can take st_low and holder can take st_high
                        cands_p = self._get_available_pharmacists_optimized(
                            [p], date, st_low, schedule_dict, hrs_now, consec, ignore_max_hours=True
                        )
                        cands_h = self._get_available_pharmacists_optimized(
                            [holder], date, st_high, schedule_dict, hrs_now, consec, ignore_max_hours=True
                        )
                        
                        if cands_p and cands_h:
                            schedule_dict[date][st_high] = holder
                            schedule_dict[date][st_low] = p
                            
                            self._revert_shift_counts(p, st_high)
                            self._update_shift_counts(p, st_low)
                            self._revert_shift_counts(holder, st_low)
                            self._update_shift_counts(holder, st_high)
                            
                            phase5_swaps += 1
                            target_found = True
                            print("  [NP-INTERNAL] %s: %s(%s->%s), %s(%s->%s)" % (
                                date.strftime('%Y-%m-%d'), p, st_high, st_low, holder, st_low, st_high))
                            break
                        else:
                            # Restore
                            schedule_dict[date][st_high] = orig_st_high_assignee
                            schedule_dict[date][st_low] = orig_st_low_assignee
                            
                    if target_found:
                        break
                if not target_found:
                    break

            print("  Phase 5 total swaps: %d" % phase5_swaps)

        # ── Rebuild final schedule DataFrame ──────────────────────────────
        final_schedule = pd.DataFrame.from_dict(schedule_dict, orient='index')
        final_schedule = final_schedule.reindex(
            columns=list(self.shift_types.keys()), fill_value='NO SHIFT'
        )
        final_schedule.fillna('NO SHIFT', inplace=True)
        final_schedule.sort_index(inplace=True)

        new_unfilled_info = {
            'problem_days': [(d, s) for d, s in still_unfilled if d in self.problem_days],
            'other_days':   [(d, s) for d, s in still_unfilled if d not in self.problem_days],
        }
        return final_schedule, new_unfilled_info


    def export_to_excel(self, schedule, unfilled_info, filename, enable_run_log=False):
        wb = Workbook()
        ws = wb.active
        ws.title = 'Monthly Schedule'
        ws_daily = wb.create_sheet("Daily Summary")
        ws_daily_codes = wb.create_sheet("Daily Summary (Codes)")
        ws_pref = wb.create_sheet("Preference Scores")
        ws_negotiate = wb.create_sheet("Negotiation Suggestions")
        ws_signature = wb.create_sheet("Signature Sheet")
        ws_min_req = wb.create_sheet("Min Req Violations")
        ws_run_logs = wb.create_sheet("Run Logs") if enable_run_log else None
        header_fill = PatternFill(start_color='FFD3D3D3', end_color='FFD3D3D3', fill_type='solid')
        border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
        ws.cell(row=1, column=1, value='Date').fill = header_fill
        for col, shift_type in enumerate(self.shift_types, 2):
            cell = ws.cell(row=1, column=col, value=f"{self.shift_types[shift_type]['description']}\n({self.shift_types[shift_type]['hours']} hrs)")
            cell.fill, cell.font, cell.alignment = header_fill, Font(bold=True), Alignment(wrap_text=True)
        # Sort the schedule by date before exporting
        schedule.sort_index(inplace=True)
        for row, date in enumerate(schedule.index, 2):
            ws.cell(row=row, column=1, value=date.strftime('%Y-%m-%d'))
            is_holiday = self.is_holiday(date)
            is_weekend = date.weekday() >= 5
            for col, shift_type in enumerate(self.shift_types, 2):
                cell = ws.cell(row=row, column=col, value=schedule.loc[date, shift_type])
                cell.border = border
                if schedule.loc[date, shift_type] == 'NO SHIFT': cell.fill = PatternFill(start_color='FFCCCCCC', fill_type='solid')
                elif is_holiday: cell.fill = PatternFill(start_color='FFFFB6C1', fill_type='solid')
                elif is_weekend: cell.fill = PatternFill(start_color='FFFFE4E1', fill_type='solid')
                elif schedule.loc[date, shift_type] == 'UNFILLED': cell.fill = PatternFill(start_color='FFFFFF00', fill_type='solid')
        ws.column_dimensions['A'].width = 22
        for col in range(2, len(self.shift_types) + 2):
            ws.column_dimensions[get_column_letter(col)].width = 20
        self.create_schedule_summaries(ws, schedule)
        self.create_daily_summary(ws_daily, schedule)
        self.create_preference_score_summary(ws_pref, schedule)
        self.create_daily_summary_with_codes(ws_daily_codes, schedule)
        self.create_negotiation_summary(ws_negotiate, schedule)

        self.create_signature_sheet(ws_signature, schedule)

        ws_min_req = wb.create_sheet("Min Req Violations")
        violations = self.validate_min_shift_requirements(schedule)

        headers = ["Pharmacist", "Department", "Required", "Actual", "Shortfall"]
        header_fill = PatternFill(start_color='FFFF0000', end_color='FFFF0000', fill_type='solid')
        border = Border(left=Side(style='thin'), right=Side(style='thin'),
                        top=Side(style='thin'), bottom=Side(style='thin'))

        for col, h in enumerate(headers, 1):
            cell = ws_min_req.cell(row=1, column=col, value=h)
            cell.fill = header_fill
            cell.font = Font(bold=True, color="FFFFFFFF")
            cell.border = border

        if not violations:
            ws_min_req.cell(row=2, column=1, value="✅ All minimum shift requirements satisfied.")
        else:
            for row, v in enumerate(violations, 2):
                for col, val in enumerate([
                    v['pharmacist'], v['department'],
                    v['required'], v['actual'], v['shortfall']
                ], 1):
                    cell = ws_min_req.cell(row=row, column=col, value=val)
                    cell.border = border
                    if v['shortfall'] > 0:
                        cell.fill = PatternFill(start_color='FFFFF2CC',
                                                end_color='FFFFF2CC', fill_type='solid')

        ws_min_req.column_dimensions['A'].width = 35
        for col in ['B','C','D','E']:
            ws_min_req.column_dimensions[col].width = 15


        if enable_run_log and ws_run_logs is not None:
            ws_run_logs.cell(row=1, column=1, value="Run Configuration").font = Font(bold=True)

            config_row = 2
            for key, value in self.run_config.items():
                ws_run_logs.cell(row=config_row, column=1, value=key)
                ws_run_logs.cell(row=config_row, column=2, value=str(value))
                config_row += 1

            log_start_row = config_row + 2
            ws_run_logs.cell(row=log_start_row, column=1, value="Run Logs").font = Font(bold=True)

            if self.run_logs:
                all_keys = []
                for log in self.run_logs:
                    for key in log.keys():
                        if key not in all_keys:
                            all_keys.append(key)

                header_row = log_start_row + 1
                for col_idx, key in enumerate(all_keys, 1):
                    cell = ws_run_logs.cell(row=header_row, column=col_idx, value=key)
                    cell.font = Font(bold=True)
                    cell.fill = PatternFill(start_color='FFD3D3D3', end_color='FFD3D3D3', fill_type='solid')
                    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

                for row_idx, log in enumerate(self.run_logs, header_row + 1):
                    for col_idx, key in enumerate(all_keys, 1):
                        ws_run_logs.cell(row=row_idx, column=col_idx, value=str(log.get(key, "")))

                for col_idx, key in enumerate(all_keys, 1):
                    ws_run_logs.column_dimensions[get_column_letter(col_idx)].width = min(max(len(str(key)) + 5, 15), 45)
            else:
                ws_run_logs.cell(row=log_start_row + 1, column=1, value="No logs recorded.")

        wb.save(filename)

    def create_signature_sheet(self, ws, schedule):
        # 1. กำหนด Style
        border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
        header_fill = PatternFill(start_color='FFD3D3D3', end_color='FFD3D3D3', fill_type='solid')
        x_fill = PatternFill(start_color='FFD3D3D3', end_color='FFD3D3D3', fill_type='solid') # สีเทาอ่อนสำหรับช่อง X
        white_fill = PatternFill(start_color='FFFFFFFF', end_color='FFFFFFFF', fill_type='solid') # สีขาวสำหรับช่องว่าง

        # Mapping สีตามกลุ่มเวรให้ตรงกับ Daily Summary 100%
        shift_colors = {
            'I100': 'FF00B050',
            'O100': 'FF00B0F0',
            'Care': 'FFD40202',
            'C8': 'FFE6B8AF',
            'I400': 'FFFF00FF',
            'O400F1': 'FF0033CC',
            'O400F2': 'FFC78AF2',
            'O400ER': 'FFED7D31',
            'ARI': 'FF7030A0',
            'Refill': 'FF741b47'
        }

        # 2. สร้าง Header แถวบนสุด (วันที่)
        ws.cell(row=1, column=1, value='Shift / Date').fill = header_fill
        ws.cell(row=1, column=1).border = border
        ws.cell(row=1, column=1).font = Font(bold=True)

        sorted_dates = sorted(schedule.index)
        for col, date in enumerate(sorted_dates, 2):
            cell = ws.cell(row=1, column=col, value=date.strftime('%d/%m'))
            cell.fill = header_fill
            cell.border = border
            cell.alignment = Alignment(horizontal='center', vertical='center')
            cell.font = Font(bold=True)

        # 3. สร้างข้อมูลแต่ละแถว (เรียงตามรหัสเวร)
        for row, shift_type in enumerate(self.shift_types.keys(), 2):
            shift_info = self.shift_types[shift_type]

            # จัด Format ชื่อเวร: ชื่อ (ชั่วโมง) \n (เวลาเริ่ม - เวลาจบ)
            shift_desc = f"{shift_info['description']} ({int(shift_info['hours'])} ชม.)\n({shift_info['start_time']} - {shift_info['end_time']})"

            # กำหนดสีพื้นหลังประจำแถวตามรหัสเวร
            row_color = 'FFFFFFFF'
            font_color = 'FF000000' # ค่าเริ่มต้นสีดำ

            # เรียง Prefix จากยาวไปสั้น เพื่อให้ระบบเช็ค O400F1, O400F2 ก่อน O400 ธรรมดา
            for prefix in sorted(shift_colors.keys(), key=len, reverse=True):
                if shift_type.startswith(prefix):
                    row_color = shift_colors[prefix]
                    # แผนกที่สีพื้นหลังเข้ม ให้ใช้ตัวหนังสือสีขาว (อิงตาม Daily Summary)
                    if prefix in ['Care', 'O400F1', 'ARI','Refill']:
                        font_color = 'FFFFFFFF'
                    break

            shift_label_fill = PatternFill(start_color=row_color, end_color=row_color, fill_type='solid')

            # ลงชื่อเวรในคอลัมน์แรก
            cell_first = ws.cell(row=row, column=1, value=shift_desc)
            cell_first.fill = shift_label_fill
            cell_first.border = border
            cell_first.alignment = Alignment(wrap_text=True, vertical='center', horizontal='left')
            cell_first.font = Font(color=font_color, bold=True)

            # 4. หยอดข้อมูล X หรือเว้นว่าง ในแต่ละวัน
            for col, date in enumerate(sorted_dates, 2):
                cell = ws.cell(row=row, column=col)
                cell.border = border
                cell.alignment = Alignment(horizontal='center', vertical='center')

                status = schedule.loc[date, shift_type]

                # ถ้าเป็นวันนั้นไม่มีเวร ให้ใส่ X และทำพื้นหลังสีเทา
                if status == 'NO SHIFT':
                    cell.value = 'X'
                    cell.fill = x_fill
                else:
                    # ถ้ามีเวร ให้ปล่อยว่างไว้เซ็นชื่อ และใช้พื้นหลังสีขาวล้วน
                    cell.value = ''
                    cell.fill = white_fill

        # 5. ปรับขนาดความกว้างของคอลัมน์
        ws.column_dimensions['A'].width = 40
        for col in range(2, len(sorted_dates) + 2):
            ws.column_dimensions[get_column_letter(col)].width = 7

    def create_negotiation_summary(self, ws, schedule):
        header_fill = PatternFill(start_color='FF4F81BD', end_color='FF4F81BD', fill_type='solid')
        border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
        bold_white_font = Font(bold=True, color="FFFFFFFF")
        alignment = Alignment(wrap_text=True, vertical='top')
        headers = ["Date", "Unfilled Shift", "Suggested Negotiation Candidates (Ranked)"]
        for col, header_text in enumerate(headers, 1):
            cell = ws.cell(row=1, column=col, value=header_text)
            cell.fill, cell.font, cell.border, cell.alignment = header_fill, bold_white_font, border, alignment
        unfilled_shifts = []
        for date in schedule.index:
            for shift_type, assigned_pharm in schedule.loc[date].items():
                if assigned_pharm == 'UNFILLED':
                    unfilled_shifts.append((date, shift_type))
        if not unfilled_shifts:
            ws.cell(row=2, column=1, value="No unfilled shifts in the final schedule.")
            return
        current_row = 2
        for date, shift_type in unfilled_shifts:
            required_skills = self.shift_types[shift_type].get('required_skills', [])
            all_candidates = []
            for p_name, p_info in self.pharmacists.items():
                if not all(skill.strip() in p_info['skills'] for skill in required_skills if skill.strip()): continue
                if len(self.get_pharmacist_shifts(p_name, date, schedule)) > 0: continue
                is_on_holiday = date.strftime('%Y-%m-%d') in p_info['holidays']
                # --- START FIX: เพิ่มตัวแปรคำนวณ % การใช้ชั่วโมง ---
                max_hrs = p_info.get('max_hours', 250)
                current_hrs = self.calculate_total_hours(p_name, schedule)

                days_in_month = pd.Timestamp(date).days_in_month
                time_elapsed_pct = date.day / days_in_month
                hours_used_pct = current_hrs / max_hrs if max_hrs > 0 else 1.0

                pharmacist_data = {
                    'name': p_name,
                    'preference_score': self.get_preference_score(p_name, shift_type),
                    'consecutive_days': self.count_consecutive_shifts(p_name, date, schedule),
                    'night_count': p_info.get('night_shift_count', 0),
                    'mixing_count': p_info.get('mixing_shift_count', 0),
                    'current_hours': current_hrs,
                    'max_hours': max_hrs,
                    'time_elapsed_pct': time_elapsed_pct,
                    'hours_used_pct': hours_used_pct
                }
                # --- END FIX ---

                suitability_score = self._calculate_suitability_score(pharmacist_data)
                suitability_score = self._calculate_suitability_score(pharmacist_data)
                all_candidates.append({'name': p_name, 'is_on_holiday': is_on_holiday, 'score': suitability_score})
            sorted_candidates = sorted(all_candidates, key=lambda x: (x['is_on_holiday'], x['score']))
            suggestions_text = []
            for i, cand in enumerate(sorted_candidates[:3]):
                status = "(On Holiday)" if cand['is_on_holiday'] else "(Available)"
                suggestions_text.append(f"{i+1}. {cand['name']} {status}")
            final_text = "\n".join(suggestions_text) if suggestions_text else "No suitable candidate found"
            ws.cell(row=current_row, column=1, value=date.strftime('%Y-%m-%d')).border = border
            ws.cell(row=current_row, column=2, value=shift_type).border = border
            cell = ws.cell(row=current_row, column=3, value=final_text)
            cell.border, cell.alignment, ws.row_dimensions[current_row].height = border, alignment, 50
            current_row += 1
        ws.column_dimensions['A'].width, ws.column_dimensions['B'].width, ws.column_dimensions['C'].width = 15, 25, 45

    def create_schedule_summaries(self, ws, schedule):
        summary_row = len(schedule) + 3
        ws.cell(row=summary_row, column=1, value="Summary").font = Font(bold=True)

        avail_row = summary_row + 2
        year  = schedule.index[0].year
        month = schedule.index[0].month
        ideal_map = self._calculate_ideal_hours(year, month)
        total_avail = sum(ideal_map.values())
        ws.cell(row=avail_row, column=1,
                value=f"Available Shift Hours ({year}-{month:02d}): {total_avail:.1f} hrs"
                ).font = Font(bold=True)
        for i, pharmacist in enumerate(self.pharmacists):
            target = ideal_map.get(pharmacist, 0)
            actual = self.calculate_total_hours(pharmacist, schedule)
            cap    = self.pharmacists[pharmacist].get('max_hours', 250)
            cap_str = f" / cap {cap}h" if cap < 250 else ""
            ws.cell(row=avail_row + i + 1, column=1, value=pharmacist)
            ws.cell(row=avail_row + i + 1, column=2,
                    value=f"Target: {target:.1f}h  |  Actual: {actual}h{cap_str}")

        hours_row = avail_row + len(self.pharmacists) + 2
        ws.cell(row=hours_row, column=1, value="Working Hours Summary").font = Font(bold=True)
        for i, pharmacist in enumerate(self.pharmacists):
            hours = self.calculate_total_hours(pharmacist, schedule)
            ws.cell(row=hours_row + i + 1, column=1, value=pharmacist)
            ws.cell(row=hours_row + i + 1, column=2, value=f"Total Hours: {hours}")
        night_row = hours_row + len(self.pharmacists) + 2
        ws.cell(row=night_row, column=1, value="Night Shift Summary").font = Font(bold=True)
        for i, pharmacist in enumerate(self.pharmacists):
            row = night_row + i + 1
            ws.cell(row=row, column=1, value=pharmacist)
            ws.cell(row=row, column=2, value=f"Night Shifts: {self.pharmacists[pharmacist]['night_shift_count']}")
        shift_row = night_row + len(self.pharmacists) + 2
        ws.cell(row=shift_row, column=1, value="Shift Count Summary").font = Font(bold=True)
        shift_types_list = list(self.shift_types.keys())
        for col_idx, shift_type in enumerate(shift_types_list, 2):
            ws.cell(row=shift_row, column=col_idx, value=shift_type).font = Font(bold=True)
        for row_idx, pharmacist in enumerate(self.pharmacists, 1):
            row_num = shift_row + row_idx
            ws.cell(row=row_num, column=1, value=pharmacist)
            for col_idx, shift_type in enumerate(shift_types_list, 2):
                count = sum(1 for d in schedule.index if schedule.loc[d, shift_type] == pharmacist)
                ws.cell(row=row_num, column=col_idx, value=count)

    def _setup_daily_summary_styles(self):
        return {
            'header_fill': PatternFill(fill_type='solid', start_color='FFD3D3D3'),
            'weekend_fill': PatternFill(fill_type='solid', start_color='FFFFE4E1'),
            'holiday_fill': PatternFill(fill_type='solid', start_color='FFFFB6C1'),
            'holiday_empty_fill': PatternFill(fill_type='solid', start_color='FFFFFF00'),
            'off_fill': PatternFill(fill_type='solid', start_color='FFD3D3D3'),
            'border': Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin')),
            'fills': {p: PatternFill(fill_type='solid', start_color=c) for p, c in [
                ('I100', 'FF00B050'), ('O100', 'FF00B0F0'), ('Care', 'FFD40202'), ('C8', 'FFE6B8AF'),
                ('I400', 'FFFF00FF'), ('O400F1', 'FF0033CC'), ('O400F2', 'FFC78AF2'),
                ('O400ER', 'FFED7D31'), ('ARI', 'FF7030A0'), ('Refill', 'FF741b47')]},
            'fonts': {
                'O400F1': Font(bold=True, color="FFFFFFFF"), 'ARI': Font(bold=True, color="FFFFFFFF"),
                'Refill': Font(bold=True, color="FFFFFFFF"),
                'default': Font(bold=True), 'header': Font(bold=True)
            }
        }

    def convert_excel_to_gsheet(self, excel_file_path, gsheet_name):
        print("\nAuthenticating and converting to Google Sheets...")
        auth.authenticate_user()
        service = build('drive', 'v3')

        file_metadata = {
            'name': gsheet_name,
            'mimeType': 'application/vnd.google-apps.spreadsheet' # คำสั่งนี้จะแปลง .xlsx เป็น Google Sheet อัตโนมัติ
        }

        media = MediaFileUpload(
            excel_file_path,
            mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
        )

        file = service.files().create(
            body=file_metadata,
            media_body=media,
            fields='id'
        ).execute()

        print(f"Successfully created Google Sheet: '{gsheet_name}' (ID: {file.get('id')})")
        return file.get('id')

    def create_daily_summary(self, ws, schedule):
        styles = self._setup_daily_summary_styles()
        ordered_pharmacists = self.get_ordered_employees()
        ws.cell(row=1, column=1, value='Pharmacist').fill = styles['header_fill']

        sorted_dates = sorted(schedule.index)

        for col, date in enumerate(sorted_dates, 2):
            cell = ws.cell(row=1, column=col, value=date.strftime('%d/%m'))
            cell.fill, cell.font = styles['header_fill'], styles['fonts']['header']
            if date.weekday() >= 5: cell.fill = styles['weekend_fill']
            if self.is_holiday(date): cell.fill = styles['holiday_fill']

        current_row = 2
        for pharmacist in ordered_pharmacists:
            if pharmacist not in self.pharmacists: continue
            ws.cell(row=current_row, column=1, value="").fill = styles['header_fill']
            ws.cell(row=current_row + 1, column=1, value=pharmacist).fill = styles['header_fill']
            ws.cell(row=current_row + 2, column=1, value="").fill = styles['header_fill']

            for col, date in enumerate(sorted_dates, 2):
                note_cell, cell1, cell2 = [ws.cell(row=current_row + r, column=col) for r in range(3)]
                all_cells = [note_cell, cell1, cell2]
                for cell in all_cells:
                    cell.border = styles['border']
                    cell.alignment = Alignment(horizontal="center", vertical="center")

                date_str = date.strftime('%Y-%m-%d')
                note_text = self.special_notes.get(pharmacist, {}).get(date_str)
                shifts = self.get_pharmacist_shifts(pharmacist, date, schedule)
                is_personal_holiday = date_str in self.pharmacists[pharmacist]['holidays']
                is_public_holiday_or_weekend = self.is_holiday(date) or date.weekday() >= 5

                # --- START OF LOGIC FIX ---
                # First, handle holiday or shift display
                if is_personal_holiday:
                    cell2.value = 'X'
                    cell1.value = None
                    for cell in all_cells:
                        cell.fill = styles['off_fill']
                else:
                    if len(shifts) > 0:
                        shift = shifts[0]
                        _st2 = self.shift_types[shift]['start_time']
                        _star2 = '*' if (hasattr(_st2, 'hour') and _st2.hour == 10 and _st2.minute == 0) or (isinstance(_st2, str) and _st2.startswith('10:00')) else ''
                        cell2.value = f"{int(self.shift_types[shift]['hours'])}N{_star2}" if self.is_night_shift(shift) else (f"{int(self.shift_types[shift]['hours'])}{_star2}" if _star2 else int(self.shift_types[shift]['hours']))
                        prefix = next((p for p in styles['fills'] if shift.startswith(p)), None)
                        if prefix:
                            fill_color = styles['fills'][prefix]
                            cell2.fill, cell2.font = fill_color, styles['fonts'].get(prefix, Font(bold=True))
                            if len(shifts) == 1: cell1.fill = fill_color

                    if len(shifts) > 1:
                        shift = shifts[1]
                        _st1 = self.shift_types[shift]['start_time']
                        _star1 = '*' if (hasattr(_st1, 'hour') and _st1.hour == 10 and _st1.minute == 0) or (isinstance(_st1, str) and _st1.startswith('10:00')) else ''
                        cell1.value = f"{int(self.shift_types[shift]['hours'])}N{_star1}" if self.is_night_shift(shift) else (f"{int(self.shift_types[shift]['hours'])}{_star1}" if _star1 else int(self.shift_types[shift]['hours']))
                        prefix = next((p for p in styles['fills'] if shift.startswith(p)), None)
                        if prefix: cell1.fill, cell1.font = styles['fills'][prefix], styles['fonts'].get(prefix, Font(bold=True))

                    if is_public_holiday_or_weekend:
                        note_cell.fill = styles['holiday_empty_fill']
                        if not shifts:
                            if not cell1.value: cell1.fill = styles['holiday_empty_fill']
                            if not cell2.value: cell2.fill = styles['holiday_empty_fill']

                # Second, ALWAYS apply the note if it exists. This overwrites nothing in note_cell
                # but ensures it is displayed regardless of holiday/shift status.
                if note_text:
                    note_cell.value = note_text
                    note_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                # --- END OF LOGIC FIX ---

            current_row += 3

        # ... (rest of the function is unchanged) ...
        total_row, unfilled_row, max_row = current_row + 1, current_row + 2, current_row + 3
        ws.cell(row=total_row, column=1, value="Total Hours").fill = styles['header_fill']
        ws.cell(row=unfilled_row, column=1, value="Unfilled Shifts").fill = styles['header_fill']
        ws.cell(row=max_row, column=1, value="Max Available Hours").fill = styles['header_fill']
        for col, date in enumerate(sorted_dates, 2):
            total_hours = sum(self.shift_types[st]['hours'] for st, p in schedule.loc[date].items() if p not in ['NO SHIFT', 'UNFILLED', 'UNASSIGNED'] and st in self.shift_types)
            max_hours_day = sum(self.shift_types[st]['hours'] for st, p in schedule.loc[date].items() if p != 'NO SHIFT' and st in self.shift_types)
            unfilled_shifts = [st for st, p in schedule.loc[date].items() if p in ['UNFILLED', 'UNASSIGNED']]
            ws.cell(row=total_row, column=col, value=total_hours).border = styles['border']
            unfilled_cell = ws.cell(row=unfilled_row, column=col)
            unfilled_cell.border = styles['border']
            if unfilled_shifts:
                unfilled_cell.value, unfilled_cell.fill = "\n".join(unfilled_shifts), PatternFill(start_color='FFFFFF00', fill_type='solid')
            else:
                unfilled_cell.value = "0"
            max_cell = ws.cell(row=max_row, column=col, value=max_hours_day)
            max_cell.border = styles['border']
        ws.column_dimensions['A'].width = 25
        for col in range(2, len(schedule.index) + 3):
            ws.column_dimensions[get_column_letter(col)].width = 7

    def create_preference_score_summary(self, ws, schedule):
        header_fill = PatternFill(start_color='FFD3D3D3', end_color='FFD3D3D3', fill_type='solid')
        border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
        bold_font = Font(bold=True)
        headers = ["Pharmacist", "Preference Score (%)", "Total Shifts Worked"]
        for col, header_text in enumerate(headers, 1):
            cell = ws.cell(row=1, column=col, value=header_text)
            cell.fill, cell.font, cell.border = header_fill, bold_font, border
        preference_scores = self.calculate_pharmacist_preference_scores(schedule)
        pharmacist_list = sorted(self.pharmacists.keys())
        for row, pharmacist in enumerate(pharmacist_list, 2):
            total_shifts = sum(1 for date in schedule.index for p in schedule.loc[date] if p == pharmacist)
            score = preference_scores.get(pharmacist, 0)
            ws.cell(row=row, column=1, value=pharmacist).border = border
            score_cell = ws.cell(row=row, column=2, value=score)
            score_cell.border = border
            score_cell.number_format = '0.00"%"'
            ws.cell(row=row, column=3, value=total_shifts).border = border
        ws.column_dimensions['A'].width, ws.column_dimensions['B'].width, ws.column_dimensions['C'].width = 30, 25, 25

    def create_daily_summary_with_codes(self, ws, schedule):
        styles = self._setup_daily_summary_styles()
        ordered_pharmacists = self.get_ordered_employees()
        ws.cell(row=1, column=1, value='Pharmacist').fill = styles['header_fill']

        sorted_dates = sorted(schedule.index)

        for col, date in enumerate(sorted_dates, 2):
            cell = ws.cell(row=1, column=col, value=date.strftime('%d/%m'))
            cell.fill, cell.font = styles['header_fill'], styles['fonts']['header']
            if date.weekday() >= 5: cell.fill = styles['weekend_fill']
            if self.is_holiday(date): cell.fill = styles['holiday_fill']

        current_row = 2
        for pharmacist in ordered_pharmacists:
            if pharmacist not in self.pharmacists: continue
            ws.cell(row=current_row, column=1, value="").fill = styles['header_fill']
            ws.cell(row=current_row + 1, column=1, value=pharmacist).fill = styles['header_fill']
            ws.cell(row=current_row + 2, column=1, value="").fill = styles['header_fill']

            for col, date in enumerate(sorted_dates, 2):
                note_cell, cell1, cell2 = [ws.cell(row=current_row + r, column=col) for r in range(3)]
                all_cells = [note_cell, cell1, cell2]
                for cell in all_cells:
                    cell.border = styles['border']
                    cell.alignment = Alignment(horizontal="center", vertical="center")
                    if cell != note_cell: cell.font = Font(bold=True, size=9)

                date_str = date.strftime('%Y-%m-%d')
                note_text = self.special_notes.get(pharmacist, {}).get(date_str)
                shifts = self.get_pharmacist_shifts(pharmacist, date, schedule)
                is_personal_holiday = date_str in self.pharmacists[pharmacist]['holidays']
                is_public_holiday_or_weekend = self.is_holiday(date) or date.weekday() >= 5

                # --- START OF LOGIC FIX ---
                # First, handle holiday or shift display
                if is_personal_holiday:
                    cell2.value = 'OFF'
                    cell1.value = None
                    for cell in all_cells:
                        cell.fill = styles['off_fill']
                else:
                    if len(shifts) > 0:
                        shift_code, cell = shifts[0], cell2
                        cell.value = shift_code
                        prefix = next((p for p in styles['fills'] if shift_code.startswith(p)), None)
                        if prefix:
                            fill_color = styles['fills'][prefix]
                            cell.fill, cell.font = fill_color, styles['fonts'].get(prefix, styles['fonts']['default'])
                            if len(shifts) == 1: cell1.fill = fill_color

                    if len(shifts) > 1:
                        shift_code, cell = shifts[1], cell1
                        cell.value = shift_code
                        prefix = next((p for p in styles['fills'] if shift_code.startswith(p)), None)
                        if prefix: cell.fill, cell.font = styles['fills'][prefix], styles['fonts'].get(prefix, styles['fonts']['default'])

                    if is_public_holiday_or_weekend:
                        note_cell.fill = styles['holiday_empty_fill']
                        if not shifts:
                            if not cell1.value: cell1.fill = styles['holiday_empty_fill']
                            if not cell2.value: cell2.fill = styles['holiday_empty_fill']

                # Second, ALWAYS apply the note if it exists.
                if note_text:
                    note_cell.value = note_text
                    note_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                # --- END OF LOGIC FIX ---

            current_row += 3

        # ... (rest of the function is unchanged) ...
        total_row, unfilled_row, max_row = current_row + 1, current_row + 2, current_row + 3
        ws.cell(row=total_row, column=1, value="Total Hours").fill = styles['header_fill']
        ws.cell(row=unfilled_row, column=1, value="Unfilled Shifts").fill = styles['header_fill']
        ws.cell(row=max_row, column=1, value="Max Available Hours").fill = styles['header_fill']
        for col, date in enumerate(sorted_dates, 2):
            total_hours = sum(self.shift_types[st]['hours'] for st, p in schedule.loc[date].items() if p not in ['NO SHIFT', 'UNFILLED', 'UNASSIGNED'] and st in self.shift_types)
            max_hours_day = sum(self.shift_types[st]['hours'] for st, p in schedule.loc[date].items() if p != 'NO SHIFT' and st in self.shift_types)
            unfilled_shifts = [st for st, p in schedule.loc[date].items() if p in ['UNFILLED', 'UNASSIGNED']]
            ws.cell(row=total_row, column=col, value=total_hours).border = styles['border']
            unfilled_cell = ws.cell(row=unfilled_row, column=col)
            unfilled_cell.border = styles['border']
            if unfilled_shifts:
                unfilled_cell.value, unfilled_cell.fill = "\n".join(unfilled_shifts), PatternFill(start_color='FFFFFF00', fill_type='solid')
            else:
                unfilled_cell.value = "0"
            max_cell = ws.cell(row=max_row, column=col, value=max_hours_day)
            max_cell.border = styles['border']
        ws.column_dimensions['A'].width = 25
        for col in range(2, len(schedule.index) + 2):
            ws.column_dimensions[get_column_letter(col)].width = 15

    def calculate_pharmacist_preference_scores(self, schedule):
        scores = {}
        MAX_POINTS_PER_SHIFT = 8
        for pharmacist in self.pharmacists:
            total_achieved_points = 0
            total_shifts_worked = 0
            for date in schedule.index:
                for shift_type, assigned_pharm in schedule.loc[date].items():
                    if assigned_pharm == pharmacist:
                        total_shifts_worked += 1
                        rank = self.get_preference_score(pharmacist, shift_type)
                        points = max(0, 9 - rank)
                        total_achieved_points += points
            if total_shifts_worked == 0:
                scores[pharmacist] = 0
            else:
                max_possible_points = total_shifts_worked * MAX_POINTS_PER_SHIFT
                if max_possible_points == 0: # Avoid division by zero
                    scores[pharmacist] = 0
                else:
                    percentage_score = (total_achieved_points / max_possible_points) * 100
                    scores[pharmacist] = percentage_score
        return scores

    # =========================================================================
    # ========== FUNCTIONS ADDED FOR SPECIFIC DATE SCHEDULING =================
    # =========================================================================

    def _pre_check_staffing_for_dates(self, dates_to_schedule):
        """
        New pre-check function that works with a list of specific dates.
        It does not modify the original _pre_check_staffing_levels function.
        """
        print("\nRunning pre-check for staffing levels for specific dates...")
        all_ok = True
        for date in dates_to_schedule:
            available_pharmacists_count = sum(1 for p_name, p_info in self.pharmacists.items()
                                              if date.strftime('%Y-%m-%d') not in p_info['holidays'])
            required_shifts_base = sum(1 for st in self.shift_types
                                       if self.is_shift_available_on_date(st, date))
            total_required_shifts_with_buffer = required_shifts_base + 3

            if available_pharmacists_count < total_required_shifts_with_buffer:
                all_ok = False
                self.problem_days.add(date)
                print(f"WARNING: Potential shortage on {date.strftime('%Y-%m-%d')}. "
                      f"Available Pharmacists: {available_pharmacists_count}, "
                      f"Required Shifts (with +3 buffer): {total_required_shifts_with_buffer}")
        if all_ok:
            print("Pre-check complete. All specified dates have sufficient staffing levels.")
        else:
            print("Pre-check complete. Identified specified dates with potential staff shortages.")
        return not all_ok

    def calculate_weekend_off_variance_for_dates(self, schedule):
        """
        New variance calculation for a specific set of dates present in a schedule.
        """
        weekend_off_counts = {p: 0 for p in self.pharmacists}
        for date in schedule.index:
            if date.weekday() >= 5: # 5 is Saturday, 6 is Sunday
                working_on_weekend = {schedule.loc[date, shift] for shift in schedule.columns if schedule.loc[date, shift] not in ['NO SHIFT', 'UNFILLED', 'UNASSIGNED']}
                for p_name in self.pharmacists:
                    if p_name not in working_on_weekend:
                        weekend_off_counts[p_name] += 1
        if len(weekend_off_counts) > 1:
            return np.var(list(weekend_off_counts.values()))
        return 0

    def calculate_metrics_for_schedule(self, schedule):
        """
        New metrics calculation function that works directly with a schedule DataFrame.
        This avoids the need to pass year and month.
        """
        hours = {p: self.calculate_total_hours(p, schedule) for p in self.pharmacists}
        night_counts = {p: self.pharmacists[p]['night_shift_count'] for p in self.pharmacists}
        weekend_off_var = self.calculate_weekend_off_variance_for_dates(schedule)
        hour_penalty = self._get_hour_imbalance_penalty(hours)

        pref_percentages = self.calculate_pharmacist_preference_scores(schedule)
        pref_variance = np.var(list(pref_percentages.values())) if len(pref_percentages) > 1 else 0

        weekend_min_off_violations = self.calculate_weekend_min_off_violations(schedule)
        metrics = {
            'hour_imbalance_penalty': hour_penalty,
            'night_variance': np.var(list(night_counts.values())) if night_counts else 0,
            'preference_score': sum(self.calculate_preference_penalty(p, schedule) for p in self.pharmacists),
            'preference_variance': pref_variance,
            'weekend_off_variance': weekend_off_var,
            'weekend_min_off_shortfall': sum(weekend_min_off_violations.values()),
            'month_segment_variance': self.calculate_month_segment_variance(schedule),
        }
        if len(hours) > 1 and len(hours.values()) > 1:
            metrics['hour_diff_for_logging'] = stdev(hours.values())
        else:
            metrics['hour_diff_for_logging'] = 0
        return metrics

    def generate_schedule_for_dates(self, dates_to_schedule, iteration_num=1):
        """
        New schedule generator that operates on a specific list of dates.
        """
        schedule_dict = {date: {shift: 'NO SHIFT' for shift in self.shift_types} for date in dates_to_schedule}
        pharmacist_hours = {p: 0 for p in self.pharmacists}
        pharmacist_consecutive_days = {p: 0 for p in self.pharmacists}

        shuffled_shifts = list(self.shift_types.keys())
        random.shuffle(shuffled_shifts)
        shuffled_pharmacists = list(self.pharmacists.keys())
        random.shuffle(shuffled_pharmacists)

        for pharmacist in self.pharmacists:
            self.pharmacists[pharmacist]['night_shift_count'] = 0
            self.pharmacists[pharmacist]['mixing_shift_count'] = 0
            self.pharmacists[pharmacist]['category_counts'] = {'Mixing': 0, 'Night': 0}

        for pharmacist, assignments in self.pre_assignments.items():
            if pharmacist not in self.pharmacists: continue
            for date_str, shift_types in assignments.items():
                date = pd.to_datetime(date_str).to_pydatetime().date()
                # Find the matching datetime object in the list
                matching_date = next((dt for dt in dates_to_schedule if dt.date() == date), None)
                if matching_date is None: continue
                for shift_type in shift_types:
                    if shift_type in self.shift_types:
                        schedule_dict[matching_date][shift_type] = pharmacist
                        self._update_shift_counts(pharmacist, shift_type)
                        pharmacist_hours[pharmacist] += self.shift_types[shift_type]['hours']

        all_dates = sorted(list(dates_to_schedule))
        problem_dates_shuffled = [d for d in all_dates if d in self.problem_days]
        random.shuffle(problem_dates_shuffled)
        other_dates_shuffled = [d for d in all_dates if d not in self.problem_days]
        random.shuffle(other_dates_shuffled)
        processing_order_dates = problem_dates_shuffled + other_dates_shuffled
        unfilled_info = {'problem_days': [], 'other_days': []}

        night_shifts_ordered = [s for s in shuffled_shifts if self.is_night_shift(s)]
        mixing_shifts_ordered = [s for s in shuffled_shifts if s.startswith('C8') and not self.is_night_shift(s)]
        care_shifts_ordered = [s for s in shuffled_shifts if s.startswith('Care') and not self.is_night_shift(s) and not s.startswith('C8')]
        other_shifts_ordered = [s for s in shuffled_shifts if not self.is_night_shift(s) and not s.startswith('C8') and not s.startswith('Care')]
        standard_shift_order = night_shifts_ordered + mixing_shifts_ordered + care_shifts_ordered + other_shifts_ordered
        problem_day_shift_order = mixing_shifts_ordered + care_shifts_ordered + night_shifts_ordered + other_shifts_ordered

        for date in tqdm(processing_order_dates, desc=f"Building Schedule (Iteration {iteration_num})", leave=False):
            previous_date = date - timedelta(days=1)
            # We can only check consecutive days if the previous day was also in our scheduling set
            if previous_date in schedule_dict:
                pharmacists_working_yesterday = {p for p in schedule_dict[previous_date].values() if p in self.pharmacists}
                for p_name in self.pharmacists:
                    if p_name in pharmacists_working_yesterday:
                        pharmacist_consecutive_days[p_name] += 1
                    else:
                        pharmacist_consecutive_days[p_name] = 0
            else: # Reset if previous day wasn't scheduled
                 for p_name in self.pharmacists:
                    pharmacist_consecutive_days[p_name] = 0

            is_day_before_problem_day = (date + timedelta(days=1)) in self.problem_days
            shifts_to_process = problem_day_shift_order if date in self.problem_days else standard_shift_order

            for shift_type in shifts_to_process:
                if schedule_dict[date][shift_type] != 'NO SHIFT' or not self.is_shift_available_on_date(shift_type, date):
                    continue
                available = self._get_available_pharmacists_optimized(shuffled_pharmacists, date, shift_type, schedule_dict, pharmacist_hours, pharmacist_consecutive_days)
                if available:
                    chosen = self._select_best_pharmacist(available, shift_type, date, is_day_before_problem_day)
                    pharmacist_to_assign = chosen['name']
                    schedule_dict[date][shift_type] = pharmacist_to_assign
                    self._update_shift_counts(pharmacist_to_assign, shift_type)
                    pharmacist_hours[pharmacist_to_assign] += self.shift_types[shift_type]['hours']
                else:
                    schedule_dict[date][shift_type] = 'UNFILLED'
                    if date in self.problem_days:
                        unfilled_info['problem_days'].append((date, shift_type))
                    else:
                        unfilled_info['other_days'].append((date, shift_type))

        final_schedule = pd.DataFrame.from_dict(schedule_dict, orient='index')
        final_schedule.fillna('NO SHIFT', inplace=True)
        return final_schedule, unfilled_info

    def optimize_schedule_for_dates(self, dates_to_schedule, iterations=10):
        """
        New optimizer for scheduling a specific list of dates.
        """
        best_schedule = None
        best_metrics = {'unfilled_problem_shifts': float('inf'), 'preference_score': float('inf')}
        best_unfilled_info = {}

        # Use the new pre-check function
        self._pre_check_staffing_for_dates(dates_to_schedule)

        print(f"\nStarting optimization for {len(dates_to_schedule)} specific dates with {iterations} iterations...")

        for i in range(iterations):
            print(f"\n--- Iteration {i+1}/{iterations} ---")
            current_schedule, unfilled_info = self.generate_schedule_for_dates(dates_to_schedule, iteration_num=i+1)

            # Use the new metrics calculation function
            metrics = self.calculate_metrics_for_schedule(current_schedule)
            metrics['unfilled_problem_shifts'] = len(unfilled_info['problem_days']) + len(unfilled_info['other_days'])

            print(f"Iteration Results -> "
                  f"Unfilled Shifts: {metrics['unfilled_problem_shifts']} | "
                  f"Hour SD: {metrics.get('hour_diff_for_logging', 0):.2f} | "
                  f"Night Var: {metrics.get('night_variance', 0):.2f} | "
                  f"Weekend Shortfall: {metrics.get('weekend_min_off_shortfall', 0)} | "
                  f"Month Segment Var: {metrics.get('month_segment_variance', 0):.2f} | "
                  f"Pref Penalty: {metrics.get('preference_score', 0):.1f}")

            if best_schedule is None or self.is_schedule_better(metrics, best_metrics):
                best_schedule = current_schedule.copy()
                best_metrics = metrics.copy()
                best_unfilled_info = unfilled_info.copy()
                print("*** Found a more balanced schedule! ***")

        if best_schedule is not None:
            print("\nOptimization complete!\nFinal metrics for the best schedule found:")
            print(f"Unfilled Shifts: {best_metrics.get('unfilled_problem_shifts', 0)} | "
                  f"Hour SD: {best_metrics.get('hour_diff_for_logging', 0):.2f} | "
                  f"Night Var: {best_metrics.get('night_variance', 0):.2f} | "
                  f"Weekend Shortfall: {best_metrics.get('weekend_min_off_shortfall', 0)} | "
                  f"Month Segment Var: {best_metrics.get('month_segment_variance', 0):.2f} | "
                  f"Pref Penalty: {best_metrics.get('preference_score', 0):.1f}")
        else:
            print("\nOptimization failed to find any valid schedule.")

        return best_schedule, best_unfilled_info

# --- Main execution block ---
if __name__ == "__main__":
    try:
        drive.mount('/content/drive')
        base_path = '/content/drive/MyDrive/Shift Sch./'

        selected_source = ask_schedule_source()
        year = ask_int_input("ใส่ปี ค.ศ. ที่ต้องการรัน", 2026, min_value=2000, max_value=2100)
        month = ask_int_input("ใส่เดือนที่ต้องการรัน", 6, min_value=1, max_value=12)
        iterations = ask_int_input("ใส่จำนวนรอบการรัน", 20, min_value=1)

        enable_run_log = ask_yes_no_input(
            "ต้องการเก็บ Log ตอนจัดเวรหรือไม่",
            default_value="N"
        )

        true_random_override = ask_yes_no_input(
            "ต้องการจัดเวรแบบ TRUE RANDOM OVERRIDE หรือไม่",
            default_value="N"
        )

        print("\nInitializing the scheduler...")
        print(f"ชนิดตาราง: {selected_source['label']}")
        print(f"Spreadsheet ID: {selected_source['spreadsheet_id']}")
        print(f"Employee sheet: {selected_source['employee_sheet_name']}")
        print(f"เดือนที่รัน: {year}-{month:02d}")
        print(f"จำนวนรอบการรัน: {iterations}")
        print(f"เก็บ Log: {'YES' if enable_run_log else 'NO'}")
        print(f"True Random Override: {'YES' if true_random_override else 'NO'}")

        input_xlsx_path = f"/content/{selected_source['output_prefix']}_input.xlsx"
        excel_file_path = download_google_sheet_as_xlsx(
            spreadsheet_id=selected_source['spreadsheet_id'],
            output_path=input_xlsx_path
        )

        scheduler = PharmacistScheduler(
            excel_file_path,
            employee_sheet_name=selected_source['employee_sheet_name'],
            staff_type=selected_source['label']
        )

        best_schedule, best_unfilled_info = scheduler.optimize_schedule(
            year=year,
            month=month,
            iterations=iterations,
            true_random_override=true_random_override,
            enable_run_log=enable_run_log
        )

        if best_schedule is not None:
            mode_suffix = "TRUE_RANDOM" if true_random_override else ""
            output_file = f"{base_path}{selected_source['output_prefix']}_{year}_{month:02d}_{mode_suffix}.xlsx"

            print(f"\nExporting the best schedule to '{output_file}'...")
            scheduler.export_to_excel(
                best_schedule,
                best_unfilled_info,
                output_file,
                enable_run_log=enable_run_log
            )

            gsheet_name = f"{selected_source['gsheet_prefix']} {year}-{month:02d} {mode_suffix}"
            scheduler.convert_excel_to_gsheet(output_file, gsheet_name)

            print("\nSuccessfully exported the schedule.")
        else:
            print("\nCould not generate a valid schedule after optimization.")

    except FileNotFoundError as e:
        print(f"\nERROR: File not found: {e}")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
from google.colab import auth
auth.authenticate_user()